# Practical 4 - Elasticsearch Query DSL

Hands-on implementation of the Elasticsearch **Query DSL** against a local
Elasticsearch 9.x node (`http://127.0.0.1:9200`), using the official
`elasticsearch` Python client.

The notebook is split into:

1. Data setup (`employees` 120 docs, `products` 60 docs).
2. Basic queries: `match_all`, `match`, `term`, `terms`, `exists`, `ids`.
3. Range queries: price, salary and date ranges.
4. Full-text search: `match_phrase`, `multi_match`, `query_string`,
   `simple_query_string`.
5. Advanced search: `fuzzy`, `prefix`, `wildcard`, `regexp`.
6. Pagination, sorting, source filtering and highlighting.
7. **Assignment - 32 business queries**, each documented with its
   requirement, query, expected output, actual output and explanation.

Every query is executed with the `run()` helper, which prints the full
documentation block and asserts exact hit counts whenever the expected result
is deterministic. Indices are dropped and recreated on every run, so the
notebook is idempotent.

**Prerequisites:** Elasticsearch running (Practical 1), `generate_data.py`
already executed, and the `p4` Jupyter kernel (see `README.md`).

## Setup - client, helpers and data

The helper functions keep every experiment consistent:

- `recreate()` / `bulk_load()` rebuild the indices from the JSON files.
- `label()` prints a short, readable line for each hit.
- `run()` performs the search and prints **Requirement / Query / Expected /
  Actual / Explanation** - the same documentation format used for the
  assignment queries.

In [1]:
import json
import re
from pathlib import Path

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk

ES_URL = "http://127.0.0.1:9200"
es = Elasticsearch(ES_URL)

info = es.info()
print("Connected to Elasticsearch", info["version"]["number"])
print("cluster:", info["cluster_name"], "| node:", info["name"])

HITS_SHOWN = 3
ASSIGNMENT_RESULTS = []


def pretty(obj):
    return json.dumps(obj, indent=2, default=str)


def hits_of(response):
    return response["hits"]["hits"]


def total_hits(response):
    return response["hits"]["total"]["value"]


def ids_of(response):
    return [hit["_id"] for hit in hits_of(response)]


def tokens(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())


def contains_phrase(text, phrase):
    text_tokens = tokens(text)
    phrase_tokens = tokens(phrase)
    width = len(phrase_tokens)
    return any(
        text_tokens[i:i + width] == phrase_tokens
        for i in range(len(text_tokens) - width + 1)
    )


def edit_distance(left, right):
    rows = list(range(len(right) + 1))
    for i, left_char in enumerate(left, start=1):
        previous, rows[0] = rows[0], i
        for j, right_char in enumerate(right, start=1):
            current = rows[j]
            rows[j] = min(
                rows[j] + 1,
                rows[j - 1] + 1,
                previous + (left_char != right_char),
            )
            previous = current
    return rows[-1]


def recreate(index, mappings):
    if es.indices.exists(index=index):
        es.indices.delete(index=index)
    es.indices.create(
        index=index,
        settings={"number_of_shards": 1, "number_of_replicas": 0},
        mappings=mappings,
    )
    print("created index:", index)


def bulk_load(index, documents, id_field):
    actions = [
        {"_index": index, "_id": str(document[id_field]), "_source": document}
        for document in documents
    ]
    indexed, errors = bulk(es, actions, refresh=True)
    suffix = f" ({len(errors)} errors)" if errors else ""
    print(f"indexed {indexed} documents into {index}{suffix}")


def label(hit, index):
    source = hit.get("_source", {}) or {}
    if index == "employees":
        parts = [hit["_id"], source.get("name"), source.get("department"),
                 source.get("city")]
        if "salary" in source:
            parts.append(f"salary {source['salary']}")
    else:
        parts = [hit["_id"], source.get("name"), source.get("category")]
        if "price" in source:
            parts.append(f"price {source['price']}")
    return " | ".join(str(part) for part in parts if part)


def run(number, title, requirement, query, expected, explanation, index,
        expected_hits=None, record=None, display=True, **search_params):
    response = es.search(index=index, query=query, **search_params)
    count = total_hits(response)

    print(f"{number} | {title}")
    print("  Requirement:")
    print("   ", requirement)
    print("  Query:")
    for line in pretty(query).splitlines():
        print("   ", line)
    if search_params:
        shown = {("from" if key == "from_" else key): value
                 for key, value in search_params.items()}
        print("  Options:")
        for line in pretty(shown).splitlines():
            print("   ", line)
    print("  Expected:")
    print("   ", expected)
    print(f"  Actual: {count} hit(s)")
    if display:
        for hit in hits_of(response)[:HITS_SHOWN]:
            print("    -", label(hit, index))
        if count > HITS_SHOWN:
            print(f"    ... {count - HITS_SHOWN} more")
    print("  Explanation:")
    print("   ", explanation)
    print("-" * 100)

    if expected_hits is not None and count != expected_hits:
        raise AssertionError(f"{number}: expected {expected_hits} hits, got {count}")
    if record:
        ASSIGNMENT_RESULTS.append({
            "query": number,
            "domain": record,
            "requirement": requirement,
            "hits": count,
        })
    return response

Connected to Elasticsearch 9.2.0
cluster: my-elasticsearch-cluster | node: my-node


### Index mappings and data load

`employees` keeps the Practical 2 schema plus two sparse fields (`email`,
`bonus`) so `exists` queries are meaningful. `products` has every field type
the queries below need (`keyword`, `text`, numeric, boolean, date, keyword
array).

In [2]:
HERE = Path.cwd()
employees_data = json.loads((HERE / "employees.json").read_text(encoding="utf-8"))
products_data = json.loads((HERE / "products.json").read_text(encoding="utf-8"))
print("employees loaded:", len(employees_data))
print("products loaded:", len(products_data))

employees_mapping = {
    "properties": {
        "employee_id": {"type": "integer"},
        "name": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
        "department": {"type": "keyword"},
        "designation": {"type": "text", "fields": {"keyword": {"type": "keyword"}}},
        "salary": {"type": "integer"},
        "city": {"type": "keyword"},
        "joining_date": {"type": "date"},
        "active": {"type": "boolean"},
        "email": {"type": "keyword"},
        "bonus": {"type": "float"},
    }
}

products_mapping = {
    "properties": {
        "product_id": {"type": "keyword"},
        "name": {"type": "text",
                 "fields": {"keyword": {"type": "keyword", "ignore_above": 256}}},
        "description": {"type": "text"},
        "brand": {"type": "keyword"},
        "category": {"type": "keyword"},
        "subcategory": {"type": "keyword"},
        "price": {"type": "float"},
        "stock": {"type": "integer"},
        "rating": {"type": "float"},
        "tags": {"type": "keyword"},
        "available": {"type": "boolean"},
        "created_at": {"type": "date"},
        "sku": {"type": "keyword"},
        "warranty_months": {"type": "integer"},
    }
}

recreate("employees", employees_mapping)
bulk_load("employees", employees_data, "employee_id")
recreate("products", products_mapping)
bulk_load("products", products_data, "product_id")

print()
print("employees count:", es.count(index="employees")["count"])
print("products count:", es.count(index="products")["count"])


def emp_where(predicate):
    return sum(1 for employee in employees_data if predicate(employee))


def prod_where(predicate):
    return sum(1 for product in products_data if predicate(product))

employees loaded: 120
products loaded: 60


created index: employees
indexed 120 documents into employees


created index: products
indexed 60 documents into products

employees count: 120
products count: 60


## Basic Queries

### 1. `match_all`

Matches every document. `size` only limits the hits that are returned; the
total match count is still reported in `hits.total.value`.

In [3]:
run(
    "Q1", "match_all - every employee",
    "Return every employee in the employees index.",
    {"match_all": {}},
    f"all {len(employees_data)} employees",
    "match_all matches every document. The default size of 10 limits the "
    "returned hits only; hits.total.value reports the full count.",
    "employees", expected_hits=len(employees_data), size=5,
)

Q1 | match_all - every employee
  Requirement:
    Return every employee in the employees index.
  Query:
    {
      "match_all": {}
    }
  Options:
    {
      "size": 5
    }
  Expected:
    all 120 employees
  Actual: 120 hit(s)
    - 101 | Nikhil Gupta | Engineering | Chennai | salary 36500
    - 102 | Gaurav Kapoor | Sales | Kolkata | salary 40500
    - 103 | Rahul Kapoor | HR | Kolkata | salary 48000
    ... 117 more
  Explanation:
    match_all matches every document. The default size of 10 limits the returned hits only; hits.total.value reports the full count.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 1, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 120, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'employees', '_id': '101', '_score': 1.0, '_source': {'employee_id': 101, 'name': 'Nikhil Gupta', 'department': 'Engineering', 'designation': 'Software Engineer', 'salary': 36500, 'city': 'Chennai', 'joining_date': '2020-09-30', 'active': True, 'email': 'nikhil.gupta101@example.com'}}, {'_index': 'employees', '_id': '102', '_score': 1.0, '_source': {'employee_id': 102, 'name': 'Gaurav Kapoor', 'department': 'Sales', 'designation': 'Sales Executive', 'salary': 40500, 'city': 'Kolkata', 'joining_date': '2018-05-11', 'active': True, 'email': 'gaurav.kapoor102@example.com'}}, {'_index': 'employees', '_id': '103', '_score': 1.0, '_source': {'employee_id': 103, 'name': 'Rahul Kapoor', 'department': 'HR', 'designation': 'HR Executive', 'salary': 48000, 'city': 'Kolkata', 'joining_date': 

### 2. `match`

`match` **analyzes** the query text with the field's analyzer. Text fields are
lowercased and split into tokens, so `rahul` matches `Rahul Kapoor`.

With `operator: and`, every analyzed term must be present (the default is
`or`, which matches if at least one term is present).

In [4]:
run(
    "Q2a", "match - single term on analyzed text",
    "Find employees whose name contains the word Rahul.",
    {"match": {"name": "rahul"}},
    f"{emp_where(lambda e: 'rahul' in tokens(e['name']))} employees named Rahul",
    "match analyzes 'rahul' into the token rahul, which matches the indexed "
    "lowercase tokens of names such as 'Rahul Kapoor'.",
    "employees",
    expected_hits=emp_where(lambda e: "rahul" in tokens(e["name"])),
)

run(
    "Q2b", "match with operator: and",
    "Find products whose name contains both wireless and headphones.",
    {"match": {"name": {"query": "wireless headphones", "operator": "and"}}},
    f"{prod_where(lambda p: {'wireless', 'headphones'} <= set(tokens(p['name'])))} product",
    "With operator 'and' both analyzed terms must occur in the field, so only "
    "'Wireless Noise-Cancelling Headphones' qualifies.",
    "products",
    expected_hits=prod_where(
        lambda p: {"wireless", "headphones"} <= set(tokens(p["name"]))
    ),
)

Q2a | match - single term on analyzed text
  Requirement:
    Find employees whose name contains the word Rahul.
  Query:
    {
      "match": {
        "name": "rahul"
      }
    }
  Expected:
    4 employees named Rahul
  Actual: 4 hit(s)
    - 103 | Rahul Kapoor | HR | Kolkata | salary 48000
    - 178 | Rahul Chowdhury | Engineering | Ahmedabad | salary 124000
    - 189 | Rahul Kapoor | Marketing | Chennai | salary 43500
    ... 1 more
  Explanation:
    match analyzes 'rahul' into the token rahul, which matches the indexed lowercase tokens of names such as 'Rahul Kapoor'.
----------------------------------------------------------------------------------------------------
Q2b | match with operator: and
  Requirement:
    Find products whose name contains both wireless and headphones.
  Query:
    {
      "match": {
        "name": {
          "query": "wireless headphones",
          "operator": "and"
        }
      }
    }
  Expected:
    1 product
  Actual: 1 hit(s)
    - P001 |

ObjectApiResponse({'took': 3, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 1, 'relation': 'eq'}, 'max_score': 6.214028, 'hits': [{'_index': 'products', '_id': 'P001', '_score': 6.214028, '_source': {'product_id': 'P001', 'name': 'Wireless Noise-Cancelling Headphones', 'description': 'Acme Wireless Noise-Cancelling Headphones with active noise cancellation and premium sound. Category: Electronics, type: Headphones.', 'brand': 'Acme', 'category': 'Electronics', 'subcategory': 'Headphones', 'price': 299.99, 'stock': 0, 'rating': 2.6, 'tags': ['wireless', 'audio', 'noise-cancelling'], 'available': False, 'created_at': '2025-08-05', 'sku': 'SKU-ELEC-0001', 'warranty_months': 12}}]}})

### 3. `term`

`term` matches the **exact indexed term** without analyzing the input. It is
the right tool for `keyword` fields and the wrong tool for `text` fields,
where the stored tokens are lowercased and split.

In [5]:
run(
    "Q3a", "term on a keyword field",
    "Find all employees in the Engineering department.",
    {"term": {"department": "Engineering"}},
    f"{emp_where(lambda e: e['department'] == 'Engineering')} Engineering employees",
    "department is a keyword field, so the exact value 'Engineering' is stored "
    "as one term and matches directly.",
    "employees",
    expected_hits=emp_where(lambda e: e["department"] == "Engineering"),
)

run(
    "Q3b", "term on an analyzed text field (negative case)",
    "Try to find employees named Rahul with a term query on name.",
    {"term": {"name": "Rahul"}},
    "0 hits - term does not analyze the query",
    "name is text, so its indexed terms are lowercase (rahul). The term query "
    "looks for the literal 'Rahul' and therefore matches nothing. This is the "
    "classic text vs keyword mistake.",
    "employees", expected_hits=0,
)

run(
    "Q3c", "term on a boolean field",
    "Find all active employees.",
    {"term": {"active": True}},
    f"{emp_where(lambda e: e['active'])} active employees",
    "Booleans are indexed as true/false terms, so term works well on them.",
    "employees", expected_hits=emp_where(lambda e: e["active"]),
)

Q3a | term on a keyword field
  Requirement:
    Find all employees in the Engineering department.
  Query:
    {
      "term": {
        "department": "Engineering"
      }
    }
  Expected:
    18 Engineering employees
  Actual: 18 hit(s)
    - 101 | Nikhil Gupta | Engineering | Chennai | salary 36500
    - 108 | Manish Mehta | Engineering | Hyderabad | salary 99500
    - 115 | Meera Sharma | Engineering | Delhi | salary 79500
    ... 15 more
  Explanation:
    department is a keyword field, so the exact value 'Engineering' is stored as one term and matches directly.
----------------------------------------------------------------------------------------------------


Q3b | term on an analyzed text field (negative case)
  Requirement:
    Try to find employees named Rahul with a term query on name.
  Query:
    {
      "term": {
        "name": "Rahul"
      }
    }
  Expected:
    0 hits - term does not analyze the query
  Actual: 0 hit(s)
  Explanation:
    name is text, so its indexed terms are lowercase (rahul). The term query looks for the literal 'Rahul' and therefore matches nothing. This is the classic text vs keyword mistake.
----------------------------------------------------------------------------------------------------


Q3c | term on a boolean field
  Requirement:
    Find all active employees.
  Query:
    {
      "term": {
        "active": true
      }
    }
  Expected:
    84 active employees
  Actual: 84 hit(s)
    - 101 | Nikhil Gupta | Engineering | Chennai | salary 36500
    - 102 | Gaurav Kapoor | Sales | Kolkata | salary 40500
    - 103 | Rahul Kapoor | HR | Kolkata | salary 48000
    ... 81 more
  Explanation:
    Booleans are indexed as true/false terms, so term works well on them.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 29, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 84, 'relation': 'eq'}, 'max_score': 0.35903898, 'hits': [{'_index': 'employees', '_id': '101', '_score': 0.35903898, '_source': {'employee_id': 101, 'name': 'Nikhil Gupta', 'department': 'Engineering', 'designation': 'Software Engineer', 'salary': 36500, 'city': 'Chennai', 'joining_date': '2020-09-30', 'active': True, 'email': 'nikhil.gupta101@example.com'}}, {'_index': 'employees', '_id': '102', '_score': 0.35903898, '_source': {'employee_id': 102, 'name': 'Gaurav Kapoor', 'department': 'Sales', 'designation': 'Sales Executive', 'salary': 40500, 'city': 'Kolkata', 'joining_date': '2018-05-11', 'active': True, 'email': 'gaurav.kapoor102@example.com'}}, {'_index': 'employees', '_id': '103', '_score': 0.35903898, '_source': {'employee_id': 103, 'name': 'Rahul Kapoor', 'department': 'HR', 'designation': 'HR Executive', 'salary': 48000, 'city':

### 4. `terms`

`terms` matches any of several values (an OR over exact terms). It works on
keyword fields and on keyword arrays such as `tags`.

In [6]:
run(
    "Q4a", "terms on department",
    "Find employees in Engineering or Sales.",
    {"terms": {"department": ["Engineering", "Sales"]}},
    f"{emp_where(lambda e: e['department'] in ('Engineering', 'Sales'))} employees",
    "terms matches documents whose department equals any listed value.",
    "employees",
    expected_hits=emp_where(lambda e: e["department"] in ("Engineering", "Sales")),
)

run(
    "Q4b", "terms on city",
    "Find employees located in Mumbai, Pune or Bengaluru.",
    {"terms": {"city": ["Mumbai", "Pune", "Bengaluru"]}},
    f"{emp_where(lambda e: e['city'] in ('Mumbai', 'Pune', 'Bengaluru'))} employees",
    "Each city is an exact keyword term; terms returns the union.",
    "employees",
    expected_hits=emp_where(
        lambda e: e["city"] in ("Mumbai", "Pune", "Bengaluru")
    ),
)

run(
    "Q4c", "terms on a keyword array",
    "Find products tagged wireless or bluetooth.",
    {"terms": {"tags": ["wireless", "bluetooth"]}},
    f"{prod_where(lambda p: set(p['tags']) & {'wireless', 'bluetooth'})} products",
    "tags is a keyword array: each element is indexed as its own term, so "
    "terms matches a document if any element equals a listed value.",
    "products",
    expected_hits=prod_where(
        lambda p: bool(set(p["tags"]) & {"wireless", "bluetooth"})
    ),
)

Q4a | terms on department
  Requirement:
    Find employees in Engineering or Sales.
  Query:
    {
      "terms": {
        "department": [
          "Engineering",
          "Sales"
        ]
      }
    }
  Expected:
    35 employees
  Actual: 35 hit(s)
    - 101 | Nikhil Gupta | Engineering | Chennai | salary 36500
    - 102 | Gaurav Kapoor | Sales | Kolkata | salary 40500
    - 108 | Manish Mehta | Engineering | Hyderabad | salary 99500
    ... 32 more
  Explanation:
    terms matches documents whose department equals any listed value.
----------------------------------------------------------------------------------------------------


Q4b | terms on city
  Requirement:
    Find employees located in Mumbai, Pune or Bengaluru.
  Query:
    {
      "terms": {
        "city": [
          "Mumbai",
          "Pune",
          "Bengaluru"
        ]
      }
    }
  Expected:
    53 employees
  Actual: 53 hit(s)
    - 107 | Deepak Patel | Operations | Pune | salary 38000
    - 109 | Shreya Reddy | Sales | Bengaluru | salary 122000
    - 110 | Meera Iyer | HR | Mumbai | salary 132500
    ... 50 more
  Explanation:
    Each city is an exact keyword term; terms returns the union.
----------------------------------------------------------------------------------------------------
Q4c | terms on a keyword array
  Requirement:
    Find products tagged wireless or bluetooth.
  Query:
    {
      "terms": {
        "tags": [
          "wireless",
          "bluetooth"
        ]
      }
    }
  Expected:
    4 products
  Actual: 4 hit(s)
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
    - P002 | Blue

ObjectApiResponse({'took': 16, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 4, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'products', '_id': 'P001', '_score': 1.0, '_source': {'product_id': 'P001', 'name': 'Wireless Noise-Cancelling Headphones', 'description': 'Acme Wireless Noise-Cancelling Headphones with active noise cancellation and premium sound. Category: Electronics, type: Headphones.', 'brand': 'Acme', 'category': 'Electronics', 'subcategory': 'Headphones', 'price': 299.99, 'stock': 0, 'rating': 2.6, 'tags': ['wireless', 'audio', 'noise-cancelling'], 'available': False, 'created_at': '2025-08-05', 'sku': 'SKU-ELEC-0001', 'warranty_months': 12}}, {'_index': 'products', '_id': 'P002', '_score': 1.0, '_source': {'product_id': 'P002', 'name': 'Bluetooth Speaker Mini', 'description': 'Nimbus Bluetooth Speaker Mini. Speakers product in Electronics. Tags: bluetooth, audio, portable.', 'brand': 'Nimbu

### 5. `exists`

`exists` matches documents that have at least one non-null value for a field.
Fields missing from a document (or set to `null`) do not match.

In [7]:
run(
    "Q5a", "exists on a populated field",
    "Find employees that have an email address.",
    {"exists": {"field": "email"}},
    f"{emp_where(lambda e: 'email' in e)} employees have an email",
    "email is intentionally sparse: only some generated records include it, so "
    "exists returns exactly those documents.",
    "employees",
    expected_hits=emp_where(lambda e: "email" in e),
)

run(
    "Q5b", "exists on a sparse numeric field",
    "Find employees that received a bonus.",
    {"exists": {"field": "bonus"}},
    f"{emp_where(lambda e: 'bonus' in e)} employees have a bonus",
    "bonus is present on roughly half the records, which makes it a good "
    "exists demo.",
    "employees",
    expected_hits=emp_where(lambda e: "bonus" in e),
)

run(
    "Q5c", "exists on a missing field",
    "Check for a field that does not exist in the mapping.",
    {"exists": {"field": "manager_id"}},
    "0 hits",
    "No document contains manager_id, so exists matches nothing (and does not "
    "raise an error).",
    "employees", expected_hits=0,
)

Q5a | exists on a populated field
  Requirement:
    Find employees that have an email address.
  Query:
    {
      "exists": {
        "field": "email"
      }
    }
  Expected:
    91 employees have an email
  Actual: 91 hit(s)
    - 101 | Nikhil Gupta | Engineering | Chennai | salary 36500
    - 102 | Gaurav Kapoor | Sales | Kolkata | salary 40500
    - 103 | Rahul Kapoor | HR | Kolkata | salary 48000
    ... 88 more
  Explanation:
    email is intentionally sparse: only some generated records include it, so exists returns exactly those documents.
----------------------------------------------------------------------------------------------------


Q5b | exists on a sparse numeric field
  Requirement:
    Find employees that received a bonus.
  Query:
    {
      "exists": {
        "field": "bonus"
      }
    }
  Expected:
    52 employees have a bonus
  Actual: 52 hit(s)
    - 104 | Vishal Reddy | Finance | Hyderabad | salary 62500
    - 105 | Manish Gupta | Marketing | Hyderabad | salary 58500
    - 108 | Manish Mehta | Engineering | Hyderabad | salary 99500
    ... 49 more
  Explanation:
    bonus is present on roughly half the records, which makes it a good exists demo.
----------------------------------------------------------------------------------------------------


Q5c | exists on a missing field
  Requirement:
    Check for a field that does not exist in the mapping.
  Query:
    {
      "exists": {
        "field": "manager_id"
      }
    }
  Expected:
    0 hits
  Actual: 0 hit(s)
  Explanation:
    No document contains manager_id, so exists matches nothing (and does not raise an error).
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 0, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 0, 'relation': 'eq'}, 'max_score': None, 'hits': []}})

### 6. `ids`

The `ids` query fetches documents by their `_id`. IDs are strings, so numeric
employee IDs are converted to strings when the documents are indexed.

In [8]:
run(
    "Q6", "ids - fetch documents by ID",
    "Retrieve employees 101, 103 and the non-existent 999.",
    {"ids": {"values": ["101", "103", "999"]}},
    "2 hits: 101 and 103",
    "ids is a direct lookup on _id. The missing ID is silently ignored, which "
    "makes it useful for batch fetches where some IDs may be stale.",
    "employees", expected_hits=2,
)

Q6 | ids - fetch documents by ID
  Requirement:
    Retrieve employees 101, 103 and the non-existent 999.
  Query:
    {
      "ids": {
        "values": [
          "101",
          "103",
          "999"
        ]
      }
    }
  Expected:
    2 hits: 101 and 103
  Actual: 2 hit(s)
    - 101 | Nikhil Gupta | Engineering | Chennai | salary 36500
    - 103 | Rahul Kapoor | HR | Kolkata | salary 48000
  Explanation:
    ids is a direct lookup on _id. The missing ID is silently ignored, which makes it useful for batch fetches where some IDs may be stale.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 1, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 2, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'employees', '_id': '101', '_score': 1.0, '_source': {'employee_id': 101, 'name': 'Nikhil Gupta', 'department': 'Engineering', 'designation': 'Software Engineer', 'salary': 36500, 'city': 'Chennai', 'joining_date': '2020-09-30', 'active': True, 'email': 'nikhil.gupta101@example.com'}}, {'_index': 'employees', '_id': '103', '_score': 1.0, '_source': {'employee_id': 103, 'name': 'Rahul Kapoor', 'department': 'HR', 'designation': 'HR Executive', 'salary': 48000, 'city': 'Kolkata', 'joining_date': '2020-06-21', 'active': True, 'email': 'rahul.kapoor103@example.com'}}]}})

## Range Queries

Range queries use `gt`, `gte`, `lt`, `lte`. They work on numeric, date and
keyword (lexicographic) fields.

### 7. Products within a price range

In [9]:
run(
    "Q7", "price range",
    "Find products priced between 500 and 3000 (inclusive).",
    {"range": {"price": {"gte": 500, "lte": 3000}}},
    f"{prod_where(lambda p: 500 <= p['price'] <= 3000)} products",
    "Both bounds are inclusive. range filters work directly on the numeric "
    "price field and are much cheaper than script filters.",
    "products",
    expected_hits=prod_where(lambda p: 500 <= p["price"] <= 3000), size=5,
)

Q7 | price range
  Requirement:
    Find products priced between 500 and 3000 (inclusive).
  Query:
    {
      "range": {
        "price": {
          "gte": 500,
          "lte": 3000
        }
      }
    }
  Options:
    {
      "size": 5
    }
  Expected:
    33 products
  Actual: 33 hit(s)
    - P006 | Wireless Charging Pad | Electronics | price 1499.0
    - P009 | Portable Power Bank 20000mAh | Electronics | price 1999.0
    - P012 | Stainless Steel Water Bottle | Home & Kitchen | price 799.0
    ... 30 more
  Explanation:
    Both bounds are inclusive. range filters work directly on the numeric price field and are much cheaper than script filters.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 3, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 33, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'products', '_id': 'P006', '_score': 1.0, '_source': {'product_id': 'P006', 'name': 'Wireless Charging Pad', 'description': 'Orbit Wireless Charging Pad. Accessories product in Electronics. Tags: wireless, charging, mobile.', 'brand': 'Orbit', 'category': 'Electronics', 'subcategory': 'Accessories', 'price': 1499.0, 'stock': 101, 'rating': 2.9, 'tags': ['wireless', 'charging', 'mobile'], 'available': True, 'created_at': '2025-11-15', 'sku': 'SKU-ELEC-0006', 'warranty_months': 12}}, {'_index': 'products', '_id': 'P009', '_score': 1.0, '_source': {'product_id': 'P009', 'name': 'Portable Power Bank 20000mAh', 'description': 'Nimbus Portable Power Bank 20000mAh. Accessories product in Electronics. Tags: powerbank, portable.', 'brand': 'Nimbus', 'category': 'Electronics', 'subcategory':

### 8. Employees earning above a salary

In [10]:
run(
    "Q8", "salary threshold",
    "Find employees earning at least 100000.",
    {"range": {"salary": {"gte": 100000}}},
    f"{emp_where(lambda e: e['salary'] >= 100000)} employees",
    "A one-sided range with gte. Salary is an integer field, so the comparison "
    "is numeric, not lexicographic.",
    "employees",
    expected_hits=emp_where(lambda e: e["salary"] >= 100000), size=5,
)

Q8 | salary threshold
  Requirement:
    Find employees earning at least 100000.
  Query:
    {
      "range": {
        "salary": {
          "gte": 100000
        }
      }
    }
  Options:
    {
      "size": 5
    }
  Expected:
    40 employees
  Actual: 40 hit(s)
    - 109 | Shreya Reddy | Sales | Bengaluru | salary 122000
    - 110 | Meera Iyer | HR | Mumbai | salary 132500
    - 113 | Manish Rao | Support | Bengaluru | salary 118500
    ... 37 more
  Explanation:
    A one-sided range with gte. Salary is an integer field, so the comparison is numeric, not lexicographic.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 1, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 40, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'employees', '_id': '109', '_score': 1.0, '_source': {'employee_id': 109, 'name': 'Shreya Reddy', 'department': 'Sales', 'designation': 'Account Manager', 'salary': 122000, 'city': 'Bengaluru', 'joining_date': '2023-03-09', 'active': True}}, {'_index': 'employees', '_id': '110', '_score': 1.0, '_source': {'employee_id': 110, 'name': 'Meera Iyer', 'department': 'HR', 'designation': 'HR Manager', 'salary': 132500, 'city': 'Mumbai', 'joining_date': '2020-07-27', 'active': False, 'bonus': 5000.0}}, {'_index': 'employees', '_id': '113', '_score': 1.0, '_source': {'employee_id': 113, 'name': 'Manish Rao', 'department': 'Support', 'designation': 'Customer Success Manager', 'salary': 118500, 'city': 'Bengaluru', 'joining_date': '2023-09-19', 'active': True, 'email': 'manish.rao113@example.

### 9. Documents created within a date range

Dates accept ISO strings (`yyyy-MM-dd`) and are inclusive at both ends. Date
math such as `now-1y` is also supported.

In [11]:
run(
    "Q9a", "product creation date range",
    "Find products created during 2024.",
    {"range": {"created_at": {"gte": "2024-01-01", "lte": "2024-12-31"}}},
    f"{prod_where(lambda p: p['created_at'][:4] == '2024')} products",
    "Elasticsearch parses the ISO strings into dates and compares them as "
    "dates, not strings.",
    "products",
    expected_hits=prod_where(lambda p: p["created_at"][:4] == "2024"), size=5,
)

run(
    "Q9b", "employee joining date range",
    "Find employees who joined during 2024.",
    {"range": {"joining_date": {"gte": "2024-01-01", "lte": "2024-12-31"}}},
    f"{emp_where(lambda e: e['joining_date'][:4] == '2024')} employees",
    "The same date range works on any date field, here joining_date.",
    "employees",
    expected_hits=emp_where(lambda e: e["joining_date"][:4] == "2024"), size=5,
)

Q9a | product creation date range
  Requirement:
    Find products created during 2024.
  Query:
    {
      "range": {
        "created_at": {
          "gte": "2024-01-01",
          "lte": "2024-12-31"
        }
      }
    }
  Options:
    {
      "size": 5
    }
  Expected:
    21 products
  Actual: 21 hit(s)
    - P012 | Stainless Steel Water Bottle | Home & Kitchen | price 799.0
    - P014 | Ergonomic Office Chair | Home & Kitchen | price 12999.0
    - P023 | Mountain Bike 21-Speed | Sports | price 18999.0
    ... 18 more
  Explanation:
    Elasticsearch parses the ISO strings into dates and compares them as dates, not strings.
----------------------------------------------------------------------------------------------------


Q9b | employee joining date range
  Requirement:
    Find employees who joined during 2024.
  Query:
    {
      "range": {
        "joining_date": {
          "gte": "2024-01-01",
          "lte": "2024-12-31"
        }
      }
    }
  Options:
    {
      "size": 5
    }
  Expected:
    16 employees
  Actual: 16 hit(s)
    - 105 | Manish Gupta | Marketing | Hyderabad | salary 58500
    - 112 | Vikram Mehta | Marketing | Pune | salary 44000
    - 118 | Vikram Rao | Finance | Mumbai | salary 111000
    ... 13 more
  Explanation:
    The same date range works on any date field, here joining_date.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 1, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 16, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'employees', '_id': '105', '_score': 1.0, '_source': {'employee_id': 105, 'name': 'Manish Gupta', 'department': 'Marketing', 'designation': 'Marketing Executive', 'salary': 58500, 'city': 'Hyderabad', 'joining_date': '2024-10-08', 'active': True, 'email': 'manish.gupta105@example.com', 'bonus': 5000.0}}, {'_index': 'employees', '_id': '112', '_score': 1.0, '_source': {'employee_id': 112, 'name': 'Vikram Mehta', 'department': 'Marketing', 'designation': 'Content Strategist', 'salary': 44000, 'city': 'Pune', 'joining_date': '2024-04-18', 'active': True, 'email': 'vikram.mehta112@example.com', 'bonus': 40000.0}}, {'_index': 'employees', '_id': '118', '_score': 1.0, '_source': {'employee_id': 118, 'name': 'Vikram Rao', 'department': 'Finance', 'designation': 'Finance Manager', 'salary'

## Full-Text Search

### 10. `match_phrase`

`match_phrase` requires the analyzed terms to appear in the same order and
adjacent to each other.

In [12]:
run(
    "Q10", "match_phrase on description",
    "Find products whose description contains the phrase noise cancellation.",
    {"match_phrase": {"description": "noise cancellation"}},
    f"{prod_where(lambda p: contains_phrase(p['description'], 'noise cancellation'))} products",
    "A match query would match either word anywhere; match_phrase additionally "
    "requires the words to be adjacent and in order.",
    "products",
    expected_hits=prod_where(
        lambda p: contains_phrase(p["description"], "noise cancellation")
    ),
)

Q10 | match_phrase on description
  Requirement:
    Find products whose description contains the phrase noise cancellation.
  Query:
    {
      "match_phrase": {
        "description": "noise cancellation"
      }
    }
  Expected:
    2 products
  Actual: 2 hit(s)
    - P007 | Noise-Cancelling Earbuds | Electronics | price 7999.0
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
  Explanation:
    A match query would match either word anywhere; match_phrase additionally requires the words to be adjacent and in order.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 1, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 2, 'relation': 'eq'}, 'max_score': 5.891617, 'hits': [{'_index': 'products', '_id': 'P007', '_score': 5.891617, '_source': {'product_id': 'P007', 'name': 'Noise-Cancelling Earbuds', 'description': 'Acme Noise-Cancelling Earbuds with active noise cancellation and premium sound. Category: Electronics, type: Headphones.', 'brand': 'Acme', 'category': 'Electronics', 'subcategory': 'Headphones', 'price': 7999.0, 'stock': 131, 'rating': 2.8, 'tags': ['audio', 'wireless', 'earbuds'], 'available': True, 'created_at': '2025-11-03', 'sku': 'SKU-ELEC-0007', 'warranty_months': 24}}, {'_index': 'products', '_id': 'P001', '_score': 5.7181253, '_source': {'product_id': 'P001', 'name': 'Wireless Noise-Cancelling Headphones', 'description': 'Acme Wireless Noise-Cancelling Headphones with active noise cancellation and premium sound. Category: Electronics, typ

### 11. `multi_match`

`multi_match` runs the same query against several fields. The default
`best_fields` mode scores the best matching field, which is ideal when one
field is likely to contain the whole query.

In [13]:
run(
    "Q11", "multi_match across name and description",
    "Search products for wireless speaker across name and description.",
    {"multi_match": {"query": "wireless speaker",
                     "fields": ["name", "description"],
                     "type": "best_fields"}},
    "products mentioning wireless or speaker, ranked by relevance",
    "multi_match expands to a bool query over the listed fields; by default "
    "any term may match, and the highest-scoring field wins.",
    "products",
    expected_hits=prod_where(
        lambda p: bool(
            (set(tokens(p["name"])) | set(tokens(p["description"])))
            & {"wireless", "speaker"}
        )
    ),
)

Q11 | multi_match across name and description
  Requirement:
    Search products for wireless speaker across name and description.
  Query:
    {
      "multi_match": {
        "query": "wireless speaker",
        "fields": [
          "name",
          "description"
        ],
        "type": "best_fields"
      }
    }
  Expected:
    products mentioning wireless or speaker, ranked by relevance
  Actual: 3 hit(s)
    - P006 | Wireless Charging Pad | Electronics | price 1499.0
    - P002 | Bluetooth Speaker Mini | Electronics | price 59.99
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
  Explanation:
    multi_match expands to a bool query over the listed fields; by default any term may match, and the highest-scoring field wins.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 2, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 3, 'relation': 'eq'}, 'max_score': 4.436035, 'hits': [{'_index': 'products', '_id': 'P006', '_score': 4.436035, '_source': {'product_id': 'P006', 'name': 'Wireless Charging Pad', 'description': 'Orbit Wireless Charging Pad. Accessories product in Electronics. Tags: wireless, charging, mobile.', 'brand': 'Orbit', 'category': 'Electronics', 'subcategory': 'Accessories', 'price': 1499.0, 'stock': 101, 'rating': 2.9, 'tags': ['wireless', 'charging', 'mobile'], 'available': True, 'created_at': '2025-11-15', 'sku': 'SKU-ELEC-0006', 'warranty_months': 12}}, {'_index': 'products', '_id': 'P002', '_score': 3.779026, '_source': {'product_id': 'P002', 'name': 'Bluetooth Speaker Mini', 'description': 'Nimbus Bluetooth Speaker Mini. Speakers product in Electronics. Tags: bluetooth, audio, portable.', 'brand': 'Nimbus', 'category': 'Electronics', 'subcate

### 12. `query_string`

`query_string` exposes a compact Lucene syntax: boolean operators, field
names, ranges, wildcards and grouping. It is powerful but strict - an invalid
syntax returns an error.

In [14]:
run(
    "Q12", "query_string with boolean logic and a range",
    "Find Engineering or Sales employees earning 80000-120000.",
    {"query_string": {
        "query": "(department:Engineering OR department:Sales) "
                 "AND salary:[80000 TO 120000]",
        "default_field": "department",
    }},
    f"{emp_where(lambda e: e['department'] in ('Engineering', 'Sales') and 80000 <= e['salary'] <= 120000)} employees",
    "query_string parses the expression server-side: grouping, OR, AND and an "
    "inclusive numeric range. department is a keyword field, so its values are "
    "matched exactly.",
    "employees",
    expected_hits=emp_where(
        lambda e: e["department"] in ("Engineering", "Sales")
        and 80000 <= e["salary"] <= 120000
    ),
)

Q12 | query_string with boolean logic and a range
  Requirement:
    Find Engineering or Sales employees earning 80000-120000.
  Query:
    {
      "query_string": {
        "query": "(department:Engineering OR department:Sales) AND salary:[80000 TO 120000]",
        "default_field": "department"
      }
    }
  Expected:
    5 employees
  Actual: 5 hit(s)
    - 193 | Shreya Menon | Sales | Kolkata | salary 119500
    - 108 | Manish Mehta | Engineering | Hyderabad | salary 99500
    - 129 | Nandini Iyer | Engineering | Kolkata | salary 112000
    ... 2 more
  Explanation:
    query_string parses the expression server-side: grouping, OR, AND and an inclusive numeric range. department is a keyword field, so its values are matched exactly.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 2, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 5, 'relation': 'eq'}, 'max_score': 2.9335895, 'hits': [{'_index': 'employees', '_id': '193', '_score': 2.9335895, '_source': {'employee_id': 193, 'name': 'Shreya Menon', 'department': 'Sales', 'designation': 'Account Manager', 'salary': 119500, 'city': 'Kolkata', 'joining_date': '2023-01-20', 'active': False, 'email': 'shreya.menon193@example.com'}}, {'_index': 'employees', '_id': '108', '_score': 2.8780198, '_source': {'employee_id': 108, 'name': 'Manish Mehta', 'department': 'Engineering', 'designation': 'Senior Software Engineer', 'salary': 99500, 'city': 'Hyderabad', 'joining_date': '2019-10-29', 'active': True, 'email': 'manish.mehta108@example.com', 'bonus': 5000.0}}, {'_index': 'employees', '_id': '129', '_score': 2.8780198, '_source': {'employee_id': 129, 'name': 'Nandini Iyer', 'department': 'Engineering', 'designation': 'Engineerin

### 13. `simple_query_string`

`simple_query_string` is a forgiving variant that never throws on user input.
It supports `+` (must), `|` (or), `-` (must not), `"` (phrase) and `*`
(prefix).

In [15]:
run(
    "Q13", "simple_query_string with must / must-not",
    "Find designations that contain engineer but not manager.",
    {"simple_query_string": {"query": "+engineer +(-manager)",
                             "fields": ["designation"]}},
    f"{emp_where(lambda e: 'engineer' in tokens(e['designation']) and 'manager' not in tokens(e['designation']))} employees",
    "A + prefix marks a required term and +(-manager) marks a required NOT "
    "group, translating into must and must_not clauses. simple_query_string "
    "never throws on malformed input, which makes it ideal for search boxes.",
    "employees",
    expected_hits=emp_where(
        lambda e: "engineer" in tokens(e["designation"])
        and "manager" not in tokens(e["designation"])
    ),
)

Q13 | simple_query_string with must / must-not
  Requirement:
    Find designations that contain engineer but not manager.
  Query:
    {
      "simple_query_string": {
        "query": "+engineer +(-manager)",
        "fields": [
          "designation"
        ]
      }
    }
  Expected:
    21 employees
  Actual: 21 hit(s)
    - 101 | Nikhil Gupta | Engineering | Chennai | salary 36500
    - 106 | Nandini Pillai | Support | Chennai | salary 40000
    - 115 | Meera Sharma | Engineering | Delhi | salary 79500
    ... 18 more
  Explanation:
    A + prefix marks a required term and +(-manager) marks a required NOT group, translating into must and must_not clauses. simple_query_string never throws on malformed input, which makes it ideal for search boxes.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 2, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 21, 'relation': 'eq'}, 'max_score': 2.736508, 'hits': [{'_index': 'employees', '_id': '101', '_score': 2.736508, '_source': {'employee_id': 101, 'name': 'Nikhil Gupta', 'department': 'Engineering', 'designation': 'Software Engineer', 'salary': 36500, 'city': 'Chennai', 'joining_date': '2020-09-30', 'active': True, 'email': 'nikhil.gupta101@example.com'}}, {'_index': 'employees', '_id': '106', '_score': 2.736508, '_source': {'employee_id': 106, 'name': 'Nandini Pillai', 'department': 'Support', 'designation': 'Support Engineer', 'salary': 40000, 'city': 'Chennai', 'joining_date': '2025-01-18', 'active': True}}, {'_index': 'employees', '_id': '115', '_score': 2.736508, '_source': {'employee_id': 115, 'name': 'Meera Sharma', 'department': 'Engineering', 'designation': 'DevOps Engineer', 'salary': 79500, 'city': 'Delhi', 'joining_date': '2025-08

## Advanced Search

### 14. `fuzzy`

`fuzzy` tolerates typos by matching terms within a Levenshtein edit distance.
`AUTO` allows 0 edits for 1-2 character terms, 1 edit for 3-5 characters and
2 edits for longer terms.

In [16]:
run(
    "Q14a", "fuzzy on a name (typo)",
    "Find employees named Rahul when the user typed rahyl.",
    {"fuzzy": {"name": {"value": "rahyl", "fuzziness": "AUTO"}}},
    "employees named Rahul (1 edit away)",
    "fuzzy is a term-level query, so it does not analyze or lowercase the "
    "input: 'rahyl' matches the indexed token 'rahul' with one edit. AUTO "
    "allows 1 edit for a 5-character term. (A capitalized 'Rahyl' would need "
    "two edits and would not match.)",
    "employees",
    expected_hits=emp_where(
        lambda e: any(edit_distance("rahyl", token) <= 1 for token in tokens(e["name"]))
    ),
)

run(
    "Q14b", "fuzzy on a product name (typo)",
    "Find headphones when the user typed headphnes.",
    {"fuzzy": {"name": {"value": "headphnes", "fuzziness": "AUTO"}}},
    f"{prod_where(lambda p: any(edit_distance('headphnes', t) <= 2 for t in tokens(p['name'])))} product",
    "The term is long enough that AUTO allows 2 edits; the missing 'o' costs "
    "one edit, so the headphones product still matches.",
    "products",
    expected_hits=prod_where(
        lambda p: any(edit_distance("headphnes", t) <= 2 for t in tokens(p["name"]))
    ),
)

Q14a | fuzzy on a name (typo)
  Requirement:
    Find employees named Rahul when the user typed rahyl.
  Query:
    {
      "fuzzy": {
        "name": {
          "value": "rahyl",
          "fuzziness": "AUTO"
        }
      }
    }
  Expected:
    employees named Rahul (1 edit away)
  Actual: 4 hit(s)
    - 103 | Rahul Kapoor | HR | Kolkata | salary 48000
    - 178 | Rahul Chowdhury | Engineering | Ahmedabad | salary 124000
    - 189 | Rahul Kapoor | Marketing | Chennai | salary 43500
    ... 1 more
  Explanation:
    fuzzy is a term-level query, so it does not analyze or lowercase the input: 'rahyl' matches the indexed token 'rahul' with one edit. AUTO allows 1 edit for a 5-character term. (A capitalized 'Rahyl' would need two edits and would not match.)
----------------------------------------------------------------------------------------------------


Q14b | fuzzy on a product name (typo)
  Requirement:
    Find headphones when the user typed headphnes.
  Query:
    {
      "fuzzy": {
        "name": {
          "value": "headphnes",
          "fuzziness": "AUTO"
        }
      }
    }
  Expected:
    1 product
  Actual: 1 hit(s)
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
  Explanation:
    The term is long enough that AUTO allows 2 edits; the missing 'o' costs one edit, so the headphones product still matches.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 41, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 1, 'relation': 'eq'}, 'max_score': 2.9662528, 'hits': [{'_index': 'products', '_id': 'P001', '_score': 2.9662528, '_source': {'product_id': 'P001', 'name': 'Wireless Noise-Cancelling Headphones', 'description': 'Acme Wireless Noise-Cancelling Headphones with active noise cancellation and premium sound. Category: Electronics, type: Headphones.', 'brand': 'Acme', 'category': 'Electronics', 'subcategory': 'Headphones', 'price': 299.99, 'stock': 0, 'rating': 2.6, 'tags': ['wireless', 'audio', 'noise-cancelling'], 'available': False, 'created_at': '2025-08-05', 'sku': 'SKU-ELEC-0001', 'warranty_months': 12}}]}})

### 15. `prefix`

`prefix` matches terms that start with the given characters. It works on
exact keyword values and on individual analyzed terms.

In [17]:
run(
    "Q15a", "prefix on a keyword city",
    "Find employees in cities starting with Beng.",
    {"prefix": {"city": "Beng"}},
    f"{emp_where(lambda e: e['city'].startswith('Beng'))} employees in Bengaluru",
    "city is a keyword field, so prefix compares against the full stored value.",
    "employees",
    expected_hits=emp_where(lambda e: e["city"].startswith("Beng")),
)

run(
    "Q15b", "prefix on analyzed text",
    "Find products whose name has a word starting with wire.",
    {"prefix": {"name": "wire"}},
    f"{prod_where(lambda p: any(t.startswith('wire') for t in tokens(p['name'])))} products",
    "On a text field prefix is applied to each indexed term, so it matches "
    "'Wireless' through its analyzed token 'wireless'.",
    "products",
    expected_hits=prod_where(
        lambda p: any(t.startswith("wire") for t in tokens(p["name"]))
    ),
)

run(
    "Q15c", "prefix on a SKU",
    "Find all Electronics products by SKU prefix.",
    {"prefix": {"sku": "SKU-ELEC"}},
    f"{prod_where(lambda p: p['sku'].startswith('SKU-ELEC'))} Electronics products",
    "SKU is a keyword with a structured prefix, the classic use case for "
    "prefix queries.",
    "products",
    expected_hits=prod_where(lambda p: p["sku"].startswith("SKU-ELEC")),
)

Q15a | prefix on a keyword city
  Requirement:
    Find employees in cities starting with Beng.
  Query:
    {
      "prefix": {
        "city": "Beng"
      }
    }
  Expected:
    14 employees in Bengaluru
  Actual: 14 hit(s)
    - 109 | Shreya Reddy | Sales | Bengaluru | salary 122000
    - 113 | Manish Rao | Support | Bengaluru | salary 118500
    - 120 | Rajesh Patel | Support | Bengaluru | salary 160000
    ... 11 more
  Explanation:
    city is a keyword field, so prefix compares against the full stored value.
----------------------------------------------------------------------------------------------------
Q15b | prefix on analyzed text
  Requirement:
    Find products whose name has a word starting with wire.
  Query:
    {
      "prefix": {
        "name": "wire"
      }
    }
  Expected:
    2 products
  Actual: 2 hit(s)
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
    - P006 | Wireless Charging Pad | Electronics | price 1499.0
  Explanati

Q15c | prefix on a SKU
  Requirement:
    Find all Electronics products by SKU prefix.
  Query:
    {
      "prefix": {
        "sku": "SKU-ELEC"
      }
    }
  Expected:
    10 Electronics products
  Actual: 10 hit(s)
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
    - P002 | Bluetooth Speaker Mini | Electronics | price 59.99
    - P003 | Smart LED TV 43 inch | Electronics | price 32999.0
    ... 7 more
  Explanation:
    SKU is a keyword with a structured prefix, the classic use case for prefix queries.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 10, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 10, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'products', '_id': 'P001', '_score': 1.0, '_source': {'product_id': 'P001', 'name': 'Wireless Noise-Cancelling Headphones', 'description': 'Acme Wireless Noise-Cancelling Headphones with active noise cancellation and premium sound. Category: Electronics, type: Headphones.', 'brand': 'Acme', 'category': 'Electronics', 'subcategory': 'Headphones', 'price': 299.99, 'stock': 0, 'rating': 2.6, 'tags': ['wireless', 'audio', 'noise-cancelling'], 'available': False, 'created_at': '2025-08-05', 'sku': 'SKU-ELEC-0001', 'warranty_months': 12}}, {'_index': 'products', '_id': 'P002', '_score': 1.0, '_source': {'product_id': 'P002', 'name': 'Bluetooth Speaker Mini', 'description': 'Nimbus Bluetooth Speaker Mini. Speakers product in Electronics. Tags: bluetooth, audio, portable.', 'brand': 'Nimb

### 16. `wildcard`

`wildcard` supports `?` (one character) and `*` (zero or more). Leading
wildcards are expensive because they cannot use the term index efficiently.

In [18]:
run(
    "Q16a", "wildcard on designation keyword",
    "Find every designation that contains Engineer.",
    {"wildcard": {"designation.keyword": {"value": "*Engineer*",
                                          "case_insensitive": True}}},
    f"{emp_where(lambda e: 'engineer' in e['designation'].lower())} employees",
    "Wildcards run against the .keyword multi-field so the whole designation is "
    "the term; case_insensitive keeps the pattern user friendly.",
    "employees",
    expected_hits=emp_where(lambda e: "engineer" in e["designation"].lower()),
)

run(
    "Q16b", "wildcard on a product name",
    "Find products whose full name contains Bottle.",
    {"wildcard": {"name.keyword": {"value": "*Bottle*",
                                   "case_insensitive": True}}},
    f"{prod_where(lambda p: 'bottle' in p['name'].lower())} products",
    "name.keyword stores the whole product name as one term, so *Bottle* is a "
    "substring search on that term.",
    "products",
    expected_hits=prod_where(lambda p: "bottle" in p["name"].lower()),
)

Q16a | wildcard on designation keyword
  Requirement:
    Find every designation that contains Engineer.
  Query:
    {
      "wildcard": {
        "designation.keyword": {
          "value": "*Engineer*",
          "case_insensitive": true
        }
      }
    }
  Expected:
    24 employees
  Actual: 24 hit(s)
    - 101 | Nikhil Gupta | Engineering | Chennai | salary 36500
    - 106 | Nandini Pillai | Support | Chennai | salary 40000
    - 108 | Manish Mehta | Engineering | Hyderabad | salary 99500
    ... 21 more
  Explanation:
    Wildcards run against the .keyword multi-field so the whole designation is the term; case_insensitive keeps the pattern user friendly.
----------------------------------------------------------------------------------------------------
Q16b | wildcard on a product name
  Requirement:
    Find products whose full name contains Bottle.
  Query:
    {
      "wildcard": {
        "name.keyword": {
          "value": "*Bottle*",
          "case_insensitive": t

ObjectApiResponse({'took': 29, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 2, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'products', '_id': 'P012', '_score': 1.0, '_source': {'product_id': 'P012', 'name': 'Stainless Steel Water Bottle', 'description': 'Globex Stainless Steel Water Bottle. Bottles product in Home & Kitchen. Tags: steel, bottle, eco.', 'brand': 'Globex', 'category': 'Home & Kitchen', 'subcategory': 'Bottles', 'price': 799.0, 'stock': 122, 'rating': 4.5, 'tags': ['steel', 'bottle', 'eco'], 'available': True, 'created_at': '2024-11-07', 'sku': 'SKU-HOME-0012', 'warranty_months': 24}}, {'_index': 'products', '_id': 'P019', '_score': 1.0, '_source': {'product_id': 'P019', 'name': 'Vacuum Insulated Water Bottle', 'description': 'Nimbus Vacuum Insulated Water Bottle. Bottles product in Home & Kitchen. Tags: flask, insulated, travel.', 'brand': 'Nimbus', 'category': 'Home & Kitchen', 'subcate

### 17. `regexp`

`regexp` matches a regular expression against the **whole term**. Patterns
anchor themselves to the start of the term, and `.*` is needed to search
inside it. Regex queries are powerful but should be used sparingly.

In [19]:
run(
    "Q17a", "regexp on a city",
    "Find employees whose city starts with Mum.",
    {"regexp": {"city": "Mum.*"}},
    f"{emp_where(lambda e: e['city'].startswith('Mum'))} employees",
    "regexp is a whole-term match, so Mum.* means 'starts with Mum followed by "
    "anything'.",
    "employees",
    expected_hits=emp_where(lambda e: e["city"].startswith("Mum")),
)

run(
    "Q17b", "regexp on an email",
    "Find employees whose email belongs to someone named Sharma.",
    {"regexp": {"email": r".*\.sharma[0-9]+@example\.com"}},
    f"{emp_where(lambda e: 'email' in e and re.fullmatch(r'.*\.sharma[0-9]+@example\.com', e['email']))} employees",
    "Emails are lowercase keywords; the pattern anchors the whole term with .* "
    "and requires lowercase 'sharma' before the numeric ID.",
    "employees",
    expected_hits=emp_where(
        lambda e: "email" in e
        and re.fullmatch(r".*\.sharma[0-9]+@example\.com", e["email"])
    ),
)

run(
    "Q17c", "regexp on a SKU",
    "Find products whose SKU category code is ELEC or BEAU.",
    {"regexp": {"sku": "SKU-(ELEC|BEAU)-[0-9]+"}},
    f"{prod_where(lambda p: p['category'] in ('Electronics', 'Beauty'))} products",
    "Alternation inside the pattern plus [0-9]+ for the numeric suffix. The "
    "pattern is matched against the full keyword term.",
    "products",
    expected_hits=prod_where(
        lambda p: p["category"] in ("Electronics", "Beauty")
    ),
)

Q17a | regexp on a city
  Requirement:
    Find employees whose city starts with Mum.
  Query:
    {
      "regexp": {
        "city": "Mum.*"
      }
    }
  Expected:
    17 employees
  Actual: 17 hit(s)
    - 110 | Meera Iyer | HR | Mumbai | salary 132500
    - 118 | Vikram Rao | Finance | Mumbai | salary 111000
    - 119 | Gaurav Joshi | Marketing | Mumbai | salary 121500
    ... 14 more
  Explanation:
    regexp is a whole-term match, so Mum.* means 'starts with Mum followed by anything'.
----------------------------------------------------------------------------------------------------


Q17b | regexp on an email
  Requirement:
    Find employees whose email belongs to someone named Sharma.
  Query:
    {
      "regexp": {
        "email": ".*\\.sharma[0-9]+@example\\.com"
      }
    }
  Expected:
    4 employees
  Actual: 4 hit(s)
    - 115 | Meera Sharma | Engineering | Delhi | salary 79500
    - 206 | Rajesh Sharma | Engineering | Ahmedabad | salary 78000
    - 208 | Sneha Sharma | HR | Pune | salary 76500
    ... 1 more
  Explanation:
    Emails are lowercase keywords; the pattern anchors the whole term with .* and requires lowercase 'sharma' before the numeric ID.
----------------------------------------------------------------------------------------------------
Q17c | regexp on a SKU
  Requirement:
    Find products whose SKU category code is ELEC or BEAU.
  Query:
    {
      "regexp": {
        "sku": "SKU-(ELEC|BEAU)-[0-9]+"
      }
    }
  Expected:
    20 products
  Actual: 20 hit(s)
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 2

ObjectApiResponse({'took': 3, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 20, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'products', '_id': 'P001', '_score': 1.0, '_source': {'product_id': 'P001', 'name': 'Wireless Noise-Cancelling Headphones', 'description': 'Acme Wireless Noise-Cancelling Headphones with active noise cancellation and premium sound. Category: Electronics, type: Headphones.', 'brand': 'Acme', 'category': 'Electronics', 'subcategory': 'Headphones', 'price': 299.99, 'stock': 0, 'rating': 2.6, 'tags': ['wireless', 'audio', 'noise-cancelling'], 'available': False, 'created_at': '2025-08-05', 'sku': 'SKU-ELEC-0001', 'warranty_months': 12}}, {'_index': 'products', '_id': 'P002', '_score': 1.0, '_source': {'product_id': 'P002', 'name': 'Bluetooth Speaker Mini', 'description': 'Nimbus Bluetooth Speaker Mini. Speakers product in Electronics. Tags: bluetooth, audio, portable.', 'brand': 'Nimbu

## Pagination, Sorting and Projection

### 18. Sort products by price

Sorting requires a field with doc values (`keyword`, numeric, date, boolean).
Text fields cannot be sorted unless fielddata is enabled.

In [20]:
ascending = run(
    "Q18a", "sort by price ascending",
    "List the cheapest products first.",
    {"match_all": {}},
    "prices in ascending order",
    "sort works on the numeric price field; _source filtering keeps the output "
    "small.",
    "products", expected_hits=len(products_data),
    sort=[{"price": "asc"}], size=5,
    source_includes=["product_id", "name", "price"],
)
ascending_prices = [hit["_source"]["price"] for hit in hits_of(ascending)]
assert ascending_prices == sorted(ascending_prices)

descending = run(
    "Q18b", "sort by price descending",
    "List the most expensive products first.",
    {"match_all": {}},
    "prices in descending order",
    "Same sort with the direction reversed.",
    "products", expected_hits=len(products_data),
    sort=[{"price": "desc"}], size=5,
    source_includes=["product_id", "name", "price"],
)
descending_prices = [hit["_source"]["price"] for hit in hits_of(descending)]
assert descending_prices == sorted(descending_prices, reverse=True)
print("sort order verified for both directions")

Q18a | sort by price ascending
  Requirement:
    List the cheapest products first.
  Query:
    {
      "match_all": {}
    }
  Options:
    {
      "sort": [
        {
          "price": "asc"
        }
      ],
      "size": 5,
      "source_includes": [
        "product_id",
        "name",
        "price"
      ]
    }
  Expected:
    prices in ascending order
  Actual: 60 hit(s)
    - P011 | Ceramic Coffee Mug | price 9.5
    - P002 | Bluetooth Speaker Mini | price 59.99
    - P058 | Aloe Vera Gel | price 299.0
    ... 57 more
  Explanation:
    sort works on the numeric price field; _source filtering keeps the output small.
----------------------------------------------------------------------------------------------------


Q18b | sort by price descending
  Requirement:
    List the most expensive products first.
  Query:
    {
      "match_all": {}
    }
  Options:
    {
      "sort": [
        {
          "price": "desc"
        }
      ],
      "size": 5,
      "source_includes": [
        "product_id",
        "name",
        "price"
      ]
    }
  Expected:
    prices in descending order
  Actual: 60 hit(s)
    - P008 | Smartphone Pro Max | price 129999.0
    - P004 | Ultra Slim Laptop 14 inch | price 78999.0
    - P030 | Foldable Treadmill | price 34999.0
    ... 57 more
  Explanation:
    Same sort with the direction reversed.
----------------------------------------------------------------------------------------------------
sort order verified for both directions


### 19. Retrieve the first 10 records

`size` controls the page size. Sorting by `product_id` makes "first" stable and
reproducible.

In [21]:
first_ten = run(
    "Q19", "first 10 records",
    "Retrieve the first 10 products by product_id.",
    {"match_all": {}},
    f"10 of {len(products_data)} documents",
    "size=10 returns ten hits while hits.total.value still reports all "
    "documents. An explicit sort keeps pagination deterministic.",
    "products", expected_hits=len(products_data),
    sort=[{"product_id": "asc"}], size=10,
)
print("returned hits:", len(hits_of(first_ten)))
assert len(hits_of(first_ten)) == 10

Q19 | first 10 records
  Requirement:
    Retrieve the first 10 products by product_id.
  Query:
    {
      "match_all": {}
    }
  Options:
    {
      "sort": [
        {
          "product_id": "asc"
        }
      ],
      "size": 10
    }
  Expected:
    10 of 60 documents
  Actual: 60 hit(s)
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
    - P002 | Bluetooth Speaker Mini | Electronics | price 59.99
    - P003 | Smart LED TV 43 inch | Electronics | price 32999.0
    ... 57 more
  Explanation:
    size=10 returns ten hits while hits.total.value still reports all documents. An explicit sort keeps pagination deterministic.
----------------------------------------------------------------------------------------------------
returned hits: 10


### 20. Implement pagination

`from` (written `from_` in Python) is the offset and `size` is the page size.
Pages are contiguous and non-overlapping. Deep pagination beyond
`index.max_result_window` (10 000 by default) requires `search_after`.

In [22]:
page_size = 5
employee_total = es.count(index="employees")["count"]
print("employees:", employee_total, "| page size:", page_size,
      "| total pages:", (employee_total + page_size - 1) // page_size)

expected_order = [
    str(employee["employee_id"])
    for employee in sorted(
        employees_data, key=lambda e: (-e["salary"], e["employee_id"])
    )
]

first_page = run(
    "Q20", "pagination - page 1 via from and size",
    "Retrieve the first page of five employees sorted by salary descending.",
    {"match_all": {}},
    f"5 hits, first of {employee_total // page_size} pages",
    "from is the offset and size is the page size. Sorting on salary with "
    "employee_id as a tie-breaker guarantees a stable order across pages, "
    "while hits.total.value still reports all matching documents.",
    "employees", expected_hits=employee_total,
    sort=[{"salary": "desc"}, {"employee_id": "asc"}],
    from_=0, size=page_size,
)

collected = ids_of(first_page)
for page in range(1, 3):
    response = es.search(
        index="employees",
        query={"match_all": {}},
        sort=[{"salary": "desc"}, {"employee_id": "asc"}],
        from_=page * page_size,
        size=page_size,
    )
    page_ids = ids_of(response)
    collected.extend(page_ids)
    print(f"page {page + 1} (from={page * page_size}):", page_ids)

assert collected == expected_order[:15]
print("pages are contiguous and non-overlapping (15 unique IDs verified)")

employees: 120 | page size: 5 | total pages: 24


Q20 | pagination - page 1 via from and size
  Requirement:
    Retrieve the first page of five employees sorted by salary descending.
  Query:
    {
      "match_all": {}
    }
  Options:
    {
      "sort": [
        {
          "salary": "desc"
        },
        {
          "employee_id": "asc"
        }
      ],
      "from": 0,
      "size": 5
    }
  Expected:
    5 hits, first of 24 pages
  Actual: 120 hit(s)
    - 120 | Rajesh Patel | Support | Bengaluru | salary 160000
    - 204 | Suresh Chowdhury | Support | Chennai | salary 155000
    - 200 | Sanjay Kapoor | Sales | Ahmedabad | salary 154500
    ... 117 more
  Explanation:
    from is the offset and size is the page size. Sorting on salary with employee_id as a tie-breaker guarantees a stable order across pages, while hits.total.value still reports all matching documents.
----------------------------------------------------------------------------------------------------


page 2 (from=5): ['181', '199', '114', '176', '152']


page 3 (from=10): ['215', '218', '144', '165', '110']
pages are contiguous and non-overlapping (15 unique IDs verified)


### 21. Source filtering

`source_includes` / `source_excludes` project the `_source` returned with each
hit. `source=False` returns only metadata, which is useful when you just need
IDs or aggregations.

In [23]:
included = run(
    "Q21a", "source filtering - includes",
    "Return only product_id, name and price for three products.",
    {"match_all": {}},
    "each _source contains exactly those three keys",
    "source_includes restricts the returned document body, reducing network "
    "and serialization cost.",
    "products", display=False,
    source_includes=["product_id", "name", "price"], size=3,
)
for hit in hits_of(included):
    print("   ", hit["_source"])
assert all(
    set(hit["_source"]) <= {"product_id", "name", "price"}
    for hit in hits_of(included)
)

excluded = run(
    "Q21b", "source filtering - excludes",
    "Return employee documents without email and bonus.",
    {"match_all": {}},
    "no _source contains email or bonus",
    "source_excludes removes sensitive or noisy fields while keeping the rest.",
    "employees", display=False, source_excludes=["email", "bonus"], size=3,
)
for hit in hits_of(excluded):
    print("   ", hit["_source"])
assert all(
    "email" not in hit["_source"] and "bonus" not in hit["_source"]
    for hit in hits_of(excluded)
)

ids_only = run(
    "Q21c", "source filtering - metadata only",
    "Return only IDs, without any _source.",
    {"match_all": {}},
    "hits have no _source field",
    "source=False is the cheapest projection and is common before a bulk "
    "fetch or reindex step.",
    "employees", display=False, source=False, size=3,
)
print("   ", ids_of(ids_only))
assert all("_source" not in hit for hit in hits_of(ids_only))

Q21a | source filtering - includes
  Requirement:
    Return only product_id, name and price for three products.
  Query:
    {
      "match_all": {}
    }
  Options:
    {
      "source_includes": [
        "product_id",
        "name",
        "price"
      ],
      "size": 3
    }
  Expected:
    each _source contains exactly those three keys
  Actual: 60 hit(s)
  Explanation:
    source_includes restricts the returned document body, reducing network and serialization cost.
----------------------------------------------------------------------------------------------------
    {'product_id': 'P001', 'name': 'Wireless Noise-Cancelling Headphones', 'price': 299.99}
    {'product_id': 'P002', 'name': 'Bluetooth Speaker Mini', 'price': 59.99}
    {'product_id': 'P003', 'name': 'Smart LED TV 43 inch', 'price': 32999.0}


Q21b | source filtering - excludes
  Requirement:
    Return employee documents without email and bonus.
  Query:
    {
      "match_all": {}
    }
  Options:
    {
      "source_excludes": [
        "email",
        "bonus"
      ],
      "size": 3
    }
  Expected:
    no _source contains email or bonus
  Actual: 120 hit(s)
  Explanation:
    source_excludes removes sensitive or noisy fields while keeping the rest.
----------------------------------------------------------------------------------------------------
    {'employee_id': 101, 'name': 'Nikhil Gupta', 'department': 'Engineering', 'designation': 'Software Engineer', 'salary': 36500, 'city': 'Chennai', 'joining_date': '2020-09-30', 'active': True}
    {'employee_id': 102, 'name': 'Gaurav Kapoor', 'department': 'Sales', 'designation': 'Sales Executive', 'salary': 40500, 'city': 'Kolkata', 'joining_date': '2018-05-11', 'active': True}
    {'employee_id': 103, 'name': 'Rahul Kapoor', 'department': 'HR', 'designation': 'HR Execu

### 22. Highlighting

Highlighting returns the matching fragments wrapped in tags (default
`<em>`), so a UI can show users why a document matched.

In [24]:
highlighted = run(
    "Q22", "highlight matches in descriptions",
    "Show where wireless occurs in product descriptions.",
    {"match": {"description": "wireless"}},
    "fragments with the matched word wrapped in <em> tags",
    "The highlighter re-analyzes the stored text and marks the query terms in "
    "the returned fragments.",
    "products", display=False, size=3,
    highlight={"fields": {"description": {
        "fragment_size": 80,
        "number_of_fragments": 1,
    }}},
)
for hit in hits_of(highlighted):
    fragments = hit.get("highlight", {}).get("description", [])
    print(f"   {hit['_id']}: {fragments}")
assert any("highlight" in hit for hit in hits_of(highlighted))

Q22 | highlight matches in descriptions
  Requirement:
    Show where wireless occurs in product descriptions.
  Query:
    {
      "match": {
        "description": "wireless"
      }
    }
  Options:
    {
      "size": 3,
      "highlight": {
        "fields": {
          "description": {
            "fragment_size": 80,
            "number_of_fragments": 1
          }
        }
      }
    }
  Expected:
    fragments with the matched word wrapped in <em> tags
  Actual: 2 hit(s)
  Explanation:
    The highlighter re-analyzes the stored text and marks the query terms in the returned fragments.
----------------------------------------------------------------------------------------------------
   P006: ['Tags: <em>wireless</em>, charging, mobile.']
   P001: ['Acme <em>Wireless</em> Noise-Cancelling Headphones with active noise cancellation and premium']


## Assignment - 32 Business Queries

Every query below is documented with the five required items:

1. **Search requirement** (the business question).
2. **Elasticsearch query** (printed as JSON, with any search options).
3. **Expected output** (including an exact hit count wherever it is
   deterministic).
4. **Actual output** (hit count and the first hits).
5. **Explanation** (why the query behaves this way).

### Domain A - Employee filtering and ranges

In [25]:
run(
    "A1", "active employees in Bengaluru",
    "HR wants active employees based in Bengaluru.",
    {"bool": {
        "filter": [
            {"term": {"active": True}},
            {"term": {"city": "Bengaluru"}},
        ]
    }},
    f"{emp_where(lambda e: e['active'] and e['city'] == 'Bengaluru')} employees",
    "bool.filter runs both exact conditions in filter context: no scoring and "
    "results can be cached.",
    "employees",
    expected_hits=emp_where(lambda e: e["active"] and e["city"] == "Bengaluru"),
    record="Employee filtering",
)

run(
    "A2", "Engineering salary band",
    "List engineers earning between 80000 and 150000.",
    {"bool": {
        "filter": [
            {"term": {"department": "Engineering"}},
            {"range": {"salary": {"gte": 80000, "lte": 150000}}},
        ]
    }},
    f"{emp_where(lambda e: e['department'] == 'Engineering' and 80000 <= e['salary'] <= 150000)} employees",
    "combines an exact keyword term with an inclusive numeric range.",
    "employees",
    expected_hits=emp_where(
        lambda e: e["department"] == "Engineering"
        and 80000 <= e["salary"] <= 150000
    ),
    record="Employee filtering",
)

run(
    "A3", "recent active hires",
    "Find active employees who joined after 2023-01-01.",
    {"bool": {
        "filter": [
            {"term": {"active": True}},
            {"range": {"joining_date": {"gt": "2023-01-01"}}},
        ]
    }},
    f"{emp_where(lambda e: e['joining_date'] > '2023-01-01' and e['active'])} employees",
    "a strict date range (gt) excludes the boundary date itself.",
    "employees",
    expected_hits=emp_where(
        lambda e: e["joining_date"] > "2023-01-01" and e["active"]
    ),
    record="Employee filtering",
)

run(
    "A4", "low earners in major cities",
    "Find employees in Mumbai, Pune or Delhi earning below 60000.",
    {"bool": {
        "filter": [
            {"terms": {"city": ["Mumbai", "Pune", "Delhi"]}},
            {"range": {"salary": {"lt": 60000}}},
        ]
    }},
    f"{emp_where(lambda e: e['city'] in ('Mumbai', 'Pune', 'Delhi') and e['salary'] < 60000)} employees",
    "terms handles the city list; lt is exclusive.",
    "employees",
    expected_hits=emp_where(
        lambda e: e["city"] in ("Mumbai", "Pune", "Delhi") and e["salary"] < 60000
    ),
    record="Employee filtering",
)

run(
    "A5", "senior high earners",
    "Find senior employees earning at least 90000.",
    {"bool": {
        "must": [{"match": {"designation": "senior"}}],
        "filter": [{"range": {"salary": {"gte": 90000}}}],
    }},
    f"{emp_where(lambda e: 'senior' in tokens(e['designation']) and e['salary'] >= 90000)} employees",
    "match is used for the analyzed designation text; the salary range is a "
    "filter so it does not affect relevance.",
    "employees",
    expected_hits=emp_where(
        lambda e: "senior" in tokens(e["designation"]) and e["salary"] >= 90000
    ),
    record="Employee filtering",
)

run(
    "A6", "active employees with a large bonus",
    "Find active employees whose bonus is above 20000.",
    {"bool": {
        "filter": [
            {"term": {"active": True}},
            {"range": {"bonus": {"gt": 20000}}},
        ]
    }},
    f"{emp_where(lambda e: e.get('bonus', 0) > 20000 and e['active'])} employees",
    "range on a sparse field only matches documents that have a bonus value.",
    "employees",
    expected_hits=emp_where(
        lambda e: e.get("bonus", 0) > 20000 and e["active"]
    ),
    record="Employee filtering",
)

A1 | active employees in Bengaluru
  Requirement:
    HR wants active employees based in Bengaluru.
  Query:
    {
      "bool": {
        "filter": [
          {
            "term": {
              "active": true
            }
          },
          {
            "term": {
              "city": "Bengaluru"
            }
          }
        ]
      }
    }
  Expected:
    9 employees
  Actual: 9 hit(s)
    - 109 | Shreya Reddy | Sales | Bengaluru | salary 122000
    - 113 | Manish Rao | Support | Bengaluru | salary 118500
    - 120 | Rajesh Patel | Support | Bengaluru | salary 160000
    ... 6 more
  Explanation:
    bool.filter runs both exact conditions in filter context: no scoring and results can be cached.
----------------------------------------------------------------------------------------------------
A2 | Engineering salary band
  Requirement:
    List engineers earning between 80000 and 150000.
  Query:
    {
      "bool": {
        "filter": [
          {
            "term"

A3 | recent active hires
  Requirement:
    Find active employees who joined after 2023-01-01.
  Query:
    {
      "bool": {
        "filter": [
          {
            "term": {
              "active": true
            }
          },
          {
            "range": {
              "joining_date": {
                "gt": "2023-01-01"
              }
            }
          }
        ]
      }
    }
  Expected:
    36 employees
  Actual: 36 hit(s)
    - 105 | Manish Gupta | Marketing | Hyderabad | salary 58500
    - 106 | Nandini Pillai | Support | Chennai | salary 40000
    - 109 | Shreya Reddy | Sales | Bengaluru | salary 122000
    ... 33 more
  Explanation:
    a strict date range (gt) excludes the boundary date itself.
----------------------------------------------------------------------------------------------------
A4 | low earners in major cities
  Requirement:
    Find employees in Mumbai, Pune or Delhi earning below 60000.
  Query:
    {
      "bool": {
        "filter": [


A5 | senior high earners
  Requirement:
    Find senior employees earning at least 90000.
  Query:
    {
      "bool": {
        "must": [
          {
            "match": {
              "designation": "senior"
            }
          }
        ],
        "filter": [
          {
            "range": {
              "salary": {
                "gte": 90000
              }
            }
          }
        ]
      }
    }
  Expected:
    4 employees
  Actual: 4 hit(s)
    - 108 | Manish Mehta | Engineering | Hyderabad | salary 99500
    - 143 | Gaurav Singh | Engineering | Mumbai | salary 106000
    - 178 | Rahul Chowdhury | Engineering | Ahmedabad | salary 124000
    ... 1 more
  Explanation:
    match is used for the analyzed designation text; the salary range is a filter so it does not affect relevance.
----------------------------------------------------------------------------------------------------
A6 | active employees with a large bonus
  Requirement:
    Find active employees 

ObjectApiResponse({'took': 17, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 14, 'relation': 'eq'}, 'max_score': 0.0, 'hits': [{'_index': 'employees', '_id': '112', '_score': 0.0, '_source': {'employee_id': 112, 'name': 'Vikram Mehta', 'department': 'Marketing', 'designation': 'Content Strategist', 'salary': 44000, 'city': 'Pune', 'joining_date': '2024-04-18', 'active': True, 'email': 'vikram.mehta112@example.com', 'bonus': 40000.0}}, {'_index': 'employees', '_id': '120', '_score': 0.0, '_source': {'employee_id': 120, 'name': 'Rajesh Patel', 'department': 'Support', 'designation': 'Support Lead', 'salary': 160000, 'city': 'Bengaluru', 'joining_date': '2019-06-10', 'active': True, 'bonus': 40000.0}}, {'_index': 'employees', '_id': '122', '_score': 0.0, '_source': {'employee_id': 122, 'name': 'Manish Rao', 'department': 'Engineering', 'designation': 'QA Engineer', 'salary': 63500, 'city': 'Ahmedabad', 'joining_date': 

### Domain B - Product catalog filters

In [26]:
run(
    "B1", "mid-range electronics and kitchen",
    "Find Electronics or Home & Kitchen products priced 1000-20000.",
    {"bool": {
        "should": [
            {"term": {"category": "Electronics"}},
            {"term": {"category": "Home & Kitchen"}},
        ],
        "minimum_should_match": 1,
        "filter": [{"range": {"price": {"gte": 1000, "lte": 20000}}}],
    }},
    f"{prod_where(lambda p: p['category'] in ('Electronics', 'Home & Kitchen') and 1000 <= p['price'] <= 20000)} products",
    "should with minimum_should_match=1 acts as an OR inside a bool query.",
    "products",
    expected_hits=prod_where(
        lambda p: p["category"] in ("Electronics", "Home & Kitchen")
        and 1000 <= p["price"] <= 20000
    ),
    record="Product catalog",
)

run(
    "B2", "highly rated in-stock products",
    "Show the best-rated in-stock products, five per page.",
    {"bool": {
        "filter": [
            {"term": {"available": True}},
            {"range": {"rating": {"gte": 4.0}}},
        ]
    }},
    f"{prod_where(lambda p: p['available'] and p['rating'] >= 4.0)} products, top 5 by rating",
    "sorts by rating and falls back to price for deterministic ordering.",
    "products",
    expected_hits=prod_where(lambda p: p["available"] and p["rating"] >= 4.0),
    sort=[{"rating": "desc"}, {"price": "asc"}], size=5,
    source_includes=["product_id", "name", "rating", "price"],
    record="Product catalog",
)

run(
    "B3", "out-of-stock products",
    "Find products that need restocking.",
    {"term": {"stock": 0}},
    f"{prod_where(lambda p: p['stock'] == 0)} products",
    "stock is an integer, so an exact term query is enough.",
    "products",
    expected_hits=prod_where(lambda p: p["stock"] == 0),
    record="Product catalog",
)

run(
    "B4", "wireless or bluetooth assortment",
    "Find products tagged wireless or bluetooth.",
    {"terms": {"tags": ["wireless", "bluetooth"]}},
    f"{prod_where(lambda p: set(p['tags']) & {'wireless', 'bluetooth'})} products",
    "terms on a keyword array returns the union of both tags.",
    "products",
    expected_hits=prod_where(
        lambda p: bool(set(p["tags"]) & {"wireless", "bluetooth"})
    ),
    record="Product catalog",
)

run(
    "B5", "cheap books",
    "Find books priced under 1000.",
    {"bool": {
        "filter": [
            {"term": {"category": "Books"}},
            {"range": {"price": {"lt": 1000}}},
        ]
    }},
    f"{prod_where(lambda p: p['category'] == 'Books' and p['price'] < 1000)} books",
    "exact category term plus an exclusive price range.",
    "products",
    expected_hits=prod_where(
        lambda p: p["category"] == "Books" and p["price"] < 1000
    ),
    record="Product catalog",
)

run(
    "B6", "long-warranty premium products",
    "Find products with at least 24 months warranty and a price above 5000.",
    {"bool": {
        "filter": [
            {"range": {"warranty_months": {"gte": 24}}},
            {"range": {"price": {"gt": 5000}}},
        ]
    }},
    f"{prod_where(lambda p: p['warranty_months'] >= 24 and p['price'] > 5000)} products",
    "multiple range filters in filter context are combined with AND.",
    "products",
    expected_hits=prod_where(
        lambda p: p["warranty_months"] >= 24 and p["price"] > 5000
    ),
    record="Product catalog",
)

B1 | mid-range electronics and kitchen


  Requirement:
    Find Electronics or Home & Kitchen products priced 1000-20000.
  Query:
    {
      "bool": {
        "should": [
          {
            "term": {
              "category": "Electronics"
            }
          },
          {
            "term": {
              "category": "Home & Kitchen"
            }
          }
        ],
        "minimum_should_match": 1,
        "filter": [
          {
            "range": {
              "price": {
                "gte": 1000,
                "lte": 20000
              }
            }
          }
        ]
      }
    }
  Expected:
    11 products
  Actual: 11 hit(s)
    - P006 | Wireless Charging Pad | Electronics | price 1499.0
    - P007 | Noise-Cancelling Earbuds | Electronics | price 7999.0
    - P009 | Portable Power Bank 20000mAh | Electronics | price 1999.0
    ... 8 more
  Explanation:
    should with minimum_should_match=1 acts as an OR inside a bool query.
----------------------------------------------------------

B5 | cheap books
  Requirement:
    Find books priced under 1000.
  Query:
    {
      "bool": {
        "filter": [
          {
            "term": {
              "category": "Books"
            }
          },
          {
            "range": {
              "price": {
                "lt": 1000
              }
            }
          }
        ]
      }
    }
  Expected:
    7 books
  Actual: 7 hit(s)
    - P031 | The Silent Horizon | Books | price 499.0
    - P032 | Data Structures Explained | Books | price 899.0
    - P034 | The Art of Cooking | Books | price 799.0
    ... 4 more
  Explanation:
    exact category term plus an exclusive price range.
----------------------------------------------------------------------------------------------------
B6 | long-warranty premium products
  Requirement:
    Find products with at least 24 months warranty and a price above 5000.
  Query:
    {
      "bool": {
        "filter": [
          {
            "range": {
              "warranty_m

ObjectApiResponse({'took': 1, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 9, 'relation': 'eq'}, 'max_score': 0.0, 'hits': [{'_index': 'products', '_id': 'P003', '_score': 0.0, '_source': {'product_id': 'P003', 'name': 'Smart LED TV 43 inch', 'description': 'Zenith Smart LED TV 43 inch. Televisions product in Electronics. Tags: tv, smart, 4k.', 'brand': 'Zenith', 'category': 'Electronics', 'subcategory': 'Televisions', 'price': 32999.0, 'stock': 132, 'rating': 4.0, 'tags': ['tv', 'smart', '4k'], 'available': True, 'created_at': '2025-02-01', 'sku': 'SKU-ELEC-0003', 'warranty_months': 24}}, {'_index': 'products', '_id': 'P004', '_score': 0.0, '_source': {'product_id': 'P004', 'name': 'Ultra Slim Laptop 14 inch', 'description': 'Vertex Ultra Slim Laptop 14 inch. Laptops product in Electronics. Tags: laptop, portable, office.', 'brand': 'Vertex', 'category': 'Electronics', 'subcategory': 'Laptops', 'price': 78999.0, '

### Domain C - Full-text and relevance

In [27]:
run(
    "C1", "exact product phrase",
    "Find the product whose name contains the phrase noise cancelling headphones.",
    {"match_phrase": {"name": "noise cancelling headphones"}},
    "the Wireless Noise-Cancelling Headphones product",
    "match_phrase keeps the three analyzed terms adjacent and in order.",
    "products",
    expected_hits=prod_where(
        lambda p: contains_phrase(p["name"], "noise cancelling headphones")
    ),
    record="Full-text search",
)

run(
    "C2", "all terms required across two fields",
    "Search products for bluetooth speaker requiring both words.",
    {"multi_match": {"query": "bluetooth speaker",
                     "fields": ["name", "description"],
                     "operator": "and"}},
    "the Bluetooth Speaker Mini product",
    "multi_match with operator=and behaves like a bool must over the terms, "
    "while still scoring each field.",
    "products",
    expected_hits=prod_where(
        lambda p: {"bluetooth", "speaker"} <= (set(tokens(p["name"])) | set(tokens(p["description"])))
    ),
    record="Full-text search",
)

run(
    "C3", "query_string across price and rating",
    "Find Electronics or Beauty products priced 1000-10000 with rating at least 4.0.",
    {"query_string": {
        "query": "(category:Electronics OR category:Beauty) "
                 "AND price:[1000 TO 10000] AND rating:>=4.0",
        "default_field": "category",
    }},
    f"{prod_where(lambda p: p['category'] in ('Electronics', 'Beauty') and 1000 <= p['price'] <= 10000 and p['rating'] >= 4.0)} products",
    "query_string mixes keyword equality, an inclusive range and a numeric "
    "comparison in one expression.",
    "products",
    expected_hits=prod_where(
        lambda p: p["category"] in ("Electronics", "Beauty")
        and 1000 <= p["price"] <= 10000
        and p["rating"] >= 4.0
    ),
    record="Full-text search",
)

run(
    "C4", "simple_query_string for a support search box",
    "A user types +wireless +headphones into a search box.",
    {"simple_query_string": {"query": "+wireless +headphones",
                             "fields": ["name", "description"]}},
    "the Wireless Noise-Cancelling Headphones product",
    "Both + terms are required, but any field may satisfy them.",
    "products",
    expected_hits=prod_where(
        lambda p: {"wireless", "headphones"} <= (set(tokens(p["name"])) | set(tokens(p["description"])))
    ),
    record="Full-text search",
)

run(
    "C5", "minimum_should_match percentage",
    "Find products matching at least 75% of the words in stainless steel water bottle.",
    {"match": {"description": {
        "query": "stainless steel water bottle",
        "minimum_should_match": "75%",
    }}},
    "the Stainless Steel Water Bottle product",
    "75% of four clauses means three terms must match, which excludes the "
    "insulated bottle that only shares water and bottle.",
    "products",
    expected_hits=prod_where(
        lambda p: sum(
            1 for term in ("stainless", "steel", "water", "bottle")
            if term in set(tokens(p["description"]))
        ) >= 3
    ),
    record="Full-text search",
)

C1 | exact product phrase
  Requirement:
    Find the product whose name contains the phrase noise cancelling headphones.
  Query:
    {
      "match_phrase": {
        "name": "noise cancelling headphones"
      }
    }
  Expected:
    the Wireless Noise-Cancelling Headphones product
  Actual: 1 hit(s)
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
  Explanation:
    match_phrase keeps the three analyzed terms adjacent and in order.
----------------------------------------------------------------------------------------------------
C2 | all terms required across two fields
  Requirement:
    Search products for bluetooth speaker requiring both words.
  Query:
    {
      "multi_match": {
        "query": "bluetooth speaker",
        "fields": [
          "name",
          "description"
        ],
        "operator": "and"
      }
    }
  Expected:
    the Bluetooth Speaker Mini product
  Actual: 1 hit(s)
    - P002 | Bluetooth Speaker Mini | Electronics

C3 | query_string across price and rating
  Requirement:
    Find Electronics or Beauty products priced 1000-10000 with rating at least 4.0.
  Query:
    {
      "query_string": {
        "query": "(category:Electronics OR category:Beauty) AND price:[1000 TO 10000] AND rating:>=4.0",
        "default_field": "category"
      }
    }
  Expected:
    3 products
  Actual: 3 hit(s)
    - P053 | Matte Lipstick Set | Beauty | price 1299.0
    - P054 | Luxury Perfume 100ml | Beauty | price 4999.0
    - P057 | Beard Trimmer Pro | Beauty | price 1799.0
  Explanation:
    query_string mixes keyword equality, an inclusive range and a numeric comparison in one expression.
----------------------------------------------------------------------------------------------------
C4 | simple_query_string for a support search box
  Requirement:
    A user types +wireless +headphones into a search box.
  Query:
    {
      "simple_query_string": {
        "query": "+wireless +headphones",
        "fields": [

C5 | minimum_should_match percentage
  Requirement:
    Find products matching at least 75% of the words in stainless steel water bottle.
  Query:
    {
      "match": {
        "description": {
          "query": "stainless steel water bottle",
          "minimum_should_match": "75%"
        }
      }
    }
  Expected:
    the Stainless Steel Water Bottle product
  Actual: 1 hit(s)
    - P012 | Stainless Steel Water Bottle | Home & Kitchen | price 799.0
  Explanation:
    75% of four clauses means three terms must match, which excludes the insulated bottle that only shares water and bottle.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 3, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 1, 'relation': 'eq'}, 'max_score': 15.046174, 'hits': [{'_index': 'products', '_id': 'P012', '_score': 15.046174, '_source': {'product_id': 'P012', 'name': 'Stainless Steel Water Bottle', 'description': 'Globex Stainless Steel Water Bottle. Bottles product in Home & Kitchen. Tags: steel, bottle, eco.', 'brand': 'Globex', 'category': 'Home & Kitchen', 'subcategory': 'Bottles', 'price': 799.0, 'stock': 122, 'rating': 4.5, 'tags': ['steel', 'bottle', 'eco'], 'available': True, 'created_at': '2024-11-07', 'sku': 'SKU-HOME-0012', 'warranty_months': 24}}]}})

### Domain D - Typo tolerance and pattern matching

In [28]:
run(
    "D1", "misspelled brand",
    "A user typed Nimbuss instead of Nimbus.",
    {"fuzzy": {"brand": {"value": "Nimbuss", "fuzziness": "AUTO"}}},
    f"{prod_where(lambda p: p['brand'] == 'Nimbus')} Nimbus products",
    "brand is a keyword, so fuzzy compares the full brand term with a one-edit "
    "budget.",
    "products",
    expected_hits=prod_where(lambda p: p["brand"] == "Nimbus"),
    record="Pattern matching",
)

run(
    "D2", "misspelled product word",
    "A user typed headphnes instead of headphones.",
    {"fuzzy": {"name": {"value": "headphnes", "fuzziness": "AUTO"}}},
    "the Headphones product",
    "fuzzy runs per analyzed token, so a single misspelled token still matches.",
    "products",
    expected_hits=prod_where(
        lambda p: any(edit_distance("headphnes", t) <= 2 for t in tokens(p["name"]))
    ),
    record="Pattern matching",
)

run(
    "D3", "SKU prefix navigation",
    "List the Electronics assortment by SKU prefix.",
    {"prefix": {"sku": "SKU-ELEC"}},
    f"{prod_where(lambda p: p['sku'].startswith('SKU-ELEC'))} products",
    "Structured SKUs make prefix queries predictable and cheap.",
    "products",
    expected_hits=prod_where(lambda p: p["sku"].startswith("SKU-ELEC")),
    record="Pattern matching",
)

run(
    "D4", "substring search in product names",
    "Find any product whose name contains Bottle.",
    {"wildcard": {"name.keyword": {"value": "*Bottle*",
                                   "case_insensitive": True}}},
    f"{prod_where(lambda p: 'bottle' in p['name'].lower())} products",
    "name.keyword holds the entire name as one term, so a leading and trailing "
    "wildcard performs a substring match.",
    "products",
    expected_hits=prod_where(lambda p: "bottle" in p["name"].lower()),
    record="Pattern matching",
)

run(
    "D5", "city regexp",
    "Find employees in any city whose name starts with Mum.",
    {"regexp": {"city": "Mum.*"}},
    f"{emp_where(lambda e: e['city'].startswith('Mum'))} employees",
    "regexp is anchored to the whole keyword term; Mum.* is effectively a "
    "starts-with test.",
    "employees",
    expected_hits=emp_where(lambda e: e["city"].startswith("Mum")),
    record="Pattern matching",
)

D1 | misspelled brand
  Requirement:
    A user typed Nimbuss instead of Nimbus.
  Query:
    {
      "fuzzy": {
        "brand": {
          "value": "Nimbuss",
          "fuzziness": "AUTO"
        }
      }
    }
  Expected:
    7 Nimbus products
  Actual: 7 hit(s)
    - P002 | Bluetooth Speaker Mini | Electronics | price 59.99
    - P009 | Portable Power Bank 20000mAh | Electronics | price 1999.0
    - P019 | Vacuum Insulated Water Bottle | Home & Kitchen | price 1299.0
    ... 4 more
  Explanation:
    brand is a keyword, so fuzzy compares the full brand term with a one-edit budget.
----------------------------------------------------------------------------------------------------


D2 | misspelled product word
  Requirement:
    A user typed headphnes instead of headphones.
  Query:
    {
      "fuzzy": {
        "name": {
          "value": "headphnes",
          "fuzziness": "AUTO"
        }
      }
    }
  Expected:
    the Headphones product
  Actual: 1 hit(s)
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
  Explanation:
    fuzzy runs per analyzed token, so a single misspelled token still matches.
----------------------------------------------------------------------------------------------------
D3 | SKU prefix navigation
  Requirement:
    List the Electronics assortment by SKU prefix.
  Query:
    {
      "prefix": {
        "sku": "SKU-ELEC"
      }
    }
  Expected:
    10 products
  Actual: 10 hit(s)
    - P001 | Wireless Noise-Cancelling Headphones | Electronics | price 299.99
    - P002 | Bluetooth Speaker Mini | Electronics | price 59.99
    - P003 | Smart LED TV 43 inch | Electronics | price 32999.0
    ... 7 more
  

D4 | substring search in product names
  Requirement:
    Find any product whose name contains Bottle.
  Query:
    {
      "wildcard": {
        "name.keyword": {
          "value": "*Bottle*",
          "case_insensitive": true
        }
      }
    }
  Expected:
    2 products
  Actual: 2 hit(s)
    - P012 | Stainless Steel Water Bottle | Home & Kitchen | price 799.0
    - P019 | Vacuum Insulated Water Bottle | Home & Kitchen | price 1299.0
  Explanation:
    name.keyword holds the entire name as one term, so a leading and trailing wildcard performs a substring match.
----------------------------------------------------------------------------------------------------
D5 | city regexp
  Requirement:
    Find employees in any city whose name starts with Mum.
  Query:
    {
      "regexp": {
        "city": "Mum.*"
      }
    }
  Expected:
    17 employees
  Actual: 17 hit(s)
    - 110 | Meera Iyer | HR | Mumbai | salary 132500
    - 118 | Vikram Rao | Finance | Mumbai | salary 111000

ObjectApiResponse({'took': 4, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 17, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'employees', '_id': '110', '_score': 1.0, '_source': {'employee_id': 110, 'name': 'Meera Iyer', 'department': 'HR', 'designation': 'HR Manager', 'salary': 132500, 'city': 'Mumbai', 'joining_date': '2020-07-27', 'active': False, 'bonus': 5000.0}}, {'_index': 'employees', '_id': '118', '_score': 1.0, '_source': {'employee_id': 118, 'name': 'Vikram Rao', 'department': 'Finance', 'designation': 'Finance Manager', 'salary': 111000, 'city': 'Mumbai', 'joining_date': '2024-09-19', 'active': True, 'email': 'vikram.rao118@example.com'}}, {'_index': 'employees', '_id': '119', '_score': 1.0, '_source': {'employee_id': 119, 'name': 'Gaurav Joshi', 'department': 'Marketing', 'designation': 'Marketing Manager', 'salary': 121500, 'city': 'Mumbai', 'joining_date': '2020-09-13', 'active': False, 'b

### Domain E - Sorting, pagination, projection and highlighting

In [29]:
expensive = run(
    "E1", "top three most expensive products",
    "Show the three most expensive products.",
    {"match_all": {}},
    "3 hits in descending price order",
    "sort by price desc with product_id as a deterministic tie-breaker.",
    "products", expected_hits=len(products_data),
    sort=[{"price": "desc"}, {"product_id": "asc"}], size=3,
    source_includes=["product_id", "name", "price"],
    record="Sorting and pagination",
)
expected_top = [
    product["product_id"]
    for product in sorted(products_data, key=lambda p: (-p["price"], p["product_id"]))
][:3]
assert ids_of(expensive) == expected_top

page_two = run(
    "E2", "second page of the salary ranking",
    "Show employees 11-20 when sorted by salary descending.",
    {"match_all": {}},
    "10 hits starting at offset 10",
    "from_=10 with size=10 skips the first page.",
    "employees", expected_hits=len(employees_data),
    sort=[{"salary": "desc"}, {"employee_id": "asc"}],
    from_=10, size=10,
    record="Sorting and pagination",
)
expected_page_two = [
    str(employee["employee_id"])
    for employee in sorted(employees_data, key=lambda e: (-e["salary"], e["employee_id"]))
][10:20]
assert ids_of(page_two) == expected_page_two

recent = run(
    "E3", "five most recently created products",
    "Show the five newest products with only ID, name and date.",
    {"match_all": {}},
    "5 hits with exactly the three projected keys",
    "created_at desc plus product_id asc for stable ordering; source_includes "
    "keeps the payload minimal.",
    "products", display=False,
    sort=[{"created_at": "desc"}, {"product_id": "asc"}], size=5,
    source_includes=["product_id", "name", "created_at"],
    record="Sorting and pagination",
)
for hit in hits_of(recent):
    print("   ", hit["_source"])
assert all(
    set(hit["_source"]) == {"product_id", "name", "created_at"}
    for hit in hits_of(recent)
)

earliest = run(
    "E4", "earliest hires without salary data",
    "Show the five earliest hires, hiding email, bonus and salary.",
    {"match_all": {}},
    "5 hits with no email, bonus or salary keys",
    "source_excludes removes sensitive/compensation fields from the response.",
    "employees", display=False,
    sort=[{"joining_date": "asc"}, {"employee_id": "asc"}], size=5,
    source_excludes=["email", "bonus", "salary"],
    record="Sorting and pagination",
)
for hit in hits_of(earliest):
    print("   ", hit["_source"])
assert all(
    not ({"email", "bonus", "salary"} & set(hit["_source"]))
    for hit in hits_of(earliest)
)

highlight = run(
    "E5", "highlight wireless in descriptions",
    "Explain why products matched the word wireless.",
    {"match": {"description": "wireless"}},
    "highlight fragments containing <em>wireless</em>",
    "The highlighter returns the matched context so a UI can display snippets.",
    "products", display=False, size=3,
    highlight={"fields": {"description": {
        "fragment_size": 80,
        "number_of_fragments": 1,
    }}},
    record="Sorting and pagination",
)
found_fragment = False
for hit in hits_of(highlight):
    fragments = hit.get("highlight", {}).get("description", [])
    if fragments:
        found_fragment = True
        print(f"   {hit['_id']}: {fragments[0]}")
assert found_fragment

E1 | top three most expensive products
  Requirement:
    Show the three most expensive products.
  Query:
    {
      "match_all": {}
    }
  Options:
    {
      "sort": [
        {
          "price": "desc"
        },
        {
          "product_id": "asc"
        }
      ],
      "size": 3,
      "source_includes": [
        "product_id",
        "name",
        "price"
      ]
    }
  Expected:
    3 hits in descending price order
  Actual: 60 hit(s)
    - P008 | Smartphone Pro Max | price 129999.0
    - P004 | Ultra Slim Laptop 14 inch | price 78999.0
    - P030 | Foldable Treadmill | price 34999.0
    ... 57 more
  Explanation:
    sort by price desc with product_id as a deterministic tie-breaker.
----------------------------------------------------------------------------------------------------
E2 | second page of the salary ranking
  Requirement:
    Show employees 11-20 when sorted by salary descending.
  Query:
    {
      "match_all": {}
    }
  Options:
    {
      "sort

E3 | five most recently created products
  Requirement:
    Show the five newest products with only ID, name and date.
  Query:
    {
      "match_all": {}
    }
  Options:
    {
      "sort": [
        {
          "created_at": "desc"
        },
        {
          "product_id": "asc"
        }
      ],
      "size": 5,
      "source_includes": [
        "product_id",
        "name",
        "created_at"
      ]
    }
  Expected:
    5 hits with exactly the three projected keys
  Actual: 60 hit(s)
  Explanation:
    created_at desc plus product_id asc for stable ordering; source_includes keeps the payload minimal.
----------------------------------------------------------------------------------------------------
    {'product_id': 'P056', 'name': 'Sunscreen SPF 50', 'created_at': '2025-12-15'}
    {'product_id': 'P010', 'name': '4K Action Camera', 'created_at': '2025-12-05'}
    {'product_id': 'P045', 'name': 'Minimalist Wrist Watch', 'created_at': '2025-11-26'}
    {'product_id': 'P

E5 | highlight wireless in descriptions
  Requirement:
    Explain why products matched the word wireless.
  Query:
    {
      "match": {
        "description": "wireless"
      }
    }
  Options:
    {
      "size": 3,
      "highlight": {
        "fields": {
          "description": {
            "fragment_size": 80,
            "number_of_fragments": 1
          }
        }
      }
    }
  Expected:
    highlight fragments containing <em>wireless</em>
  Actual: 2 hit(s)
  Explanation:
    The highlighter returns the matched context so a UI can display snippets.
----------------------------------------------------------------------------------------------------
   P006: Tags: <em>wireless</em>, charging, mobile.
   P001: Acme <em>Wireless</em> Noise-Cancelling Headphones with active noise cancellation and premium


### Domain F - Combined business rules

In [30]:
run(
    "F1", "active engineers and finance staff in metro cities",
    "Find active Engineering or Finance employees earning at least 80000 in "
    "Bengaluru, Hyderabad, Pune or Mumbai.",
    {"bool": {
        "filter": [
            {"term": {"active": True}},
            {"terms": {"department": ["Engineering", "Finance"]}},
            {"range": {"salary": {"gte": 80000}}},
            {"terms": {"city": ["Bengaluru", "Hyderabad", "Pune", "Mumbai"]}},
        ]
    }},
    f"{emp_where(lambda e: e['active'] and e['department'] in ('Engineering', 'Finance') and e['salary'] >= 80000 and e['city'] in ('Bengaluru', 'Hyderabad', 'Pune', 'Mumbai'))} employees",
    "Four filter clauses are ANDed inside a bool query; filter context keeps "
    "the query fast and cacheable.",
    "employees",
    expected_hits=emp_where(
        lambda e: e["active"]
        and e["department"] in ("Engineering", "Finance")
        and e["salary"] >= 80000
        and e["city"] in ("Bengaluru", "Hyderabad", "Pune", "Mumbai")
    ),
    record="Combined business rules",
)

run(
    "F2", "affordable highly rated electronics and beauty",
    "Find available Electronics or Beauty products rated 4.5+ and priced at "
    "most 5000.",
    {"bool": {
        "filter": [
            {"term": {"available": True}},
            {"terms": {"category": ["Electronics", "Beauty"]}},
            {"range": {"rating": {"gte": 4.5}}},
            {"range": {"price": {"lte": 5000}}},
        ]
    }},
    f"{prod_where(lambda p: p['available'] and p['category'] in ('Electronics', 'Beauty') and p['rating'] >= 4.5 and p['price'] <= 5000)} products",
    "availability, category, rating and price are combined as filters.",
    "products",
    expected_hits=prod_where(
        lambda p: p["available"]
        and p["category"] in ("Electronics", "Beauty")
        and p["rating"] >= 4.5
        and p["price"] <= 5000
    ),
    record="Combined business rules",
)

run(
    "F3", "well-paid active managers",
    "Find active managers earning more than 90000.",
    {"bool": {
        "must": [{"match": {"designation": "manager"}}],
        "filter": [
            {"range": {"salary": {"gt": 90000}}},
            {"term": {"active": True}},
        ],
    }},
    f"{emp_where(lambda e: 'manager' in tokens(e['designation']) and e['salary'] > 90000 and e['active'])} employees",
    "match on analyzed designation (manager) plus exact filters for salary and "
    "active status.",
    "employees",
    expected_hits=emp_where(
        lambda e: "manager" in tokens(e["designation"])
        and e["salary"] > 90000
        and e["active"]
    ),
    record="Combined business rules",
)

run(
    "F4", "cheap non-book non-fashion items",
    "Find in-stock products under 500 outside Books and Fashion.",
    {"bool": {
        "must_not": [{"terms": {"category": ["Books", "Fashion"]}}],
        "filter": [
            {"range": {"price": {"lt": 500}}},
            {"range": {"stock": {"gt": 0}}},
        ],
    }},
    f"{prod_where(lambda p: p['category'] not in ('Books', 'Fashion') and p['price'] < 500 and p['stock'] > 0)} products",
    "must_not excludes both categories while the filters require a low price "
    "and positive stock.",
    "products",
    expected_hits=prod_where(
        lambda p: p["category"] not in ("Books", "Fashion")
        and p["price"] < 500
        and p["stock"] > 0
    ),
    record="Combined business rules",
)

run(
    "F5", "2024 hires with a bonus outside HR",
    "Find active employees hired in 2024 who have a bonus and are not in HR.",
    {"bool": {
        "filter": [
            {"range": {"joining_date": {"gte": "2024-01-01",
                                        "lte": "2024-12-31"}}},
            {"term": {"active": True}},
            {"exists": {"field": "bonus"}},
        ],
        "must_not": [{"term": {"department": "HR"}}],
    }},
    f"{emp_where(lambda e: e['joining_date'][:4] == '2024' and e['active'] and 'bonus' in e and e['department'] != 'HR')} employees",
    "Date range, boolean term and exists are ANDed, then HR is excluded with "
    "must_not.",
    "employees",
    expected_hits=emp_where(
        lambda e: e["joining_date"][:4] == "2024"
        and e["active"]
        and "bonus" in e
        and e["department"] != "HR"
    ),
    record="Combined business rules",
)

F1 | active engineers and finance staff in metro cities
  Requirement:
    Find active Engineering or Finance employees earning at least 80000 in Bengaluru, Hyderabad, Pune or Mumbai.
  Query:
    {
      "bool": {
        "filter": [
          {
            "term": {
              "active": true
            }
          },
          {
            "terms": {
              "department": [
                "Engineering",
                "Finance"
              ]
            }
          },
          {
            "range": {
              "salary": {
                "gte": 80000
              }
            }
          },
          {
            "terms": {
              "city": [
                "Bengaluru",
                "Hyderabad",
                "Pune",
                "Mumbai"
              ]
            }
          }
        ]
      }
    }
  Expected:
    6 employees
  Actual: 6 hit(s)
    - 108 | Manish Mehta | Engineering | Hyderabad | salary 99500
    - 118 | Vikram Rao | Finance

F2 | affordable highly rated electronics and beauty
  Requirement:
    Find available Electronics or Beauty products rated 4.5+ and priced at most 5000.
  Query:
    {
      "bool": {
        "filter": [
          {
            "term": {
              "available": true
            }
          },
          {
            "terms": {
              "category": [
                "Electronics",
                "Beauty"
              ]
            }
          },
          {
            "range": {
              "rating": {
                "gte": 4.5
              }
            }
          },
          {
            "range": {
              "price": {
                "lte": 5000
              }
            }
          }
        ]
      }
    }
  Expected:
    3 products
  Actual: 3 hit(s)
    - P054 | Luxury Perfume 100ml | Beauty | price 4999.0
    - P056 | Sunscreen SPF 50 | Beauty | price 649.0
    - P057 | Beard Trimmer Pro | Beauty | price 1799.0
  Explanation:
    availability, category, r

F4 | cheap non-book non-fashion items
  Requirement:
    Find in-stock products under 500 outside Books and Fashion.
  Query:
    {
      "bool": {
        "must_not": [
          {
            "terms": {
              "category": [
                "Books",
                "Fashion"
              ]
            }
          }
        ],
        "filter": [
          {
            "range": {
              "price": {
                "lt": 500
              }
            }
          },
          {
            "range": {
              "stock": {
                "gt": 0
              }
            }
          }
        ]
      }
    }
  Expected:
    5 products
  Actual: 5 hit(s)
    - P002 | Bluetooth Speaker Mini | Electronics | price 59.99
    - P011 | Ceramic Coffee Mug | Home & Kitchen | price 9.5
    - P017 | Bamboo Cutting Board | Home & Kitchen | price 499.0
    ... 2 more
  Explanation:
    must_not excludes both categories while the filters require a low price and positive stock.
--

F5 | 2024 hires with a bonus outside HR
  Requirement:
    Find active employees hired in 2024 who have a bonus and are not in HR.
  Query:
    {
      "bool": {
        "filter": [
          {
            "range": {
              "joining_date": {
                "gte": "2024-01-01",
                "lte": "2024-12-31"
              }
            }
          },
          {
            "term": {
              "active": true
            }
          },
          {
            "exists": {
              "field": "bonus"
            }
          }
        ],
        "must_not": [
          {
            "term": {
              "department": "HR"
            }
          }
        ]
      }
    }
  Expected:
    5 employees
  Actual: 5 hit(s)
    - 105 | Manish Gupta | Marketing | Hyderabad | salary 58500
    - 112 | Vikram Mehta | Marketing | Pune | salary 44000
    - 165 | Neha Mehta | Sales | Hyderabad | salary 134000
    ... 2 more
  Explanation:
    Date range, boolean term and exists are

ObjectApiResponse({'took': 4, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 5, 'relation': 'eq'}, 'max_score': 0.0, 'hits': [{'_index': 'employees', '_id': '105', '_score': 0.0, '_source': {'employee_id': 105, 'name': 'Manish Gupta', 'department': 'Marketing', 'designation': 'Marketing Executive', 'salary': 58500, 'city': 'Hyderabad', 'joining_date': '2024-10-08', 'active': True, 'email': 'manish.gupta105@example.com', 'bonus': 5000.0}}, {'_index': 'employees', '_id': '112', '_score': 0.0, '_source': {'employee_id': 112, 'name': 'Vikram Mehta', 'department': 'Marketing', 'designation': 'Content Strategist', 'salary': 44000, 'city': 'Pune', 'joining_date': '2024-04-18', 'active': True, 'email': 'vikram.mehta112@example.com', 'bonus': 40000.0}}, {'_index': 'employees', '_id': '165', '_score': 0.0, '_source': {'employee_id': 165, 'name': 'Neha Mehta', 'department': 'Sales', 'designation': 'Account Manager', 'salary': 1

### Assignment summary

The `run()` helper recorded every assignment query. The table below is the
index of all documented business queries.

In [31]:
print(f"{'Query':<7}{'Matches':>8}  {'Domain':<26}Requirement")
print("-" * 110)
for row in ASSIGNMENT_RESULTS:
    print(f"{row['query']:<7}{row['hits']:>5}  {row['domain']:<26}{row['requirement'][:70]}")
print()
print("documented assignment queries:", len(ASSIGNMENT_RESULTS))
assert len(ASSIGNMENT_RESULTS) >= 30

Query   Matches  Domain                    Requirement
--------------------------------------------------------------------------------------------------------------
A1         9  Employee filtering        HR wants active employees based in Bengaluru.
A2         7  Employee filtering        List engineers earning between 80000 and 150000.
A3        36  Employee filtering        Find active employees who joined after 2023-01-01.
A4        20  Employee filtering        Find employees in Mumbai, Pune or Delhi earning below 60000.
A5         4  Employee filtering        Find senior employees earning at least 90000.
A6        14  Employee filtering        Find active employees whose bonus is above 20000.
B1        11  Product catalog           Find Electronics or Home & Kitchen products priced 1000-20000.
B2        20  Product catalog           Show the best-rated in-stock products, five per page.
B3         5  Product catalog           Find products that need restocking.
B4         4  Prod

### Sanity checks - why some queries correctly return zero

Two deliberately zero-result queries show the traps from Task 3 in a business
context. They are not part of the 32 assignment queries.

In [32]:
run(
    "N1", "negative test - term on analyzed text",
    "Try to match the exact string 'Rahul' with a term query on name.",
    {"term": {"name": "Rahul"}},
    "0 hits",
    "name is text: its terms are lowercase, so the capitalized literal does "
    "not exist in the index.",
    "employees", expected_hits=0,
)

run(
    "N2", "negative test - term case sensitivity on keyword",
    "Try to match the category 'electronics' in lowercase.",
    {"term": {"category": "electronics"}},
    "0 hits",
    "keyword terms are case-sensitive; the indexed value is 'Electronics'.",
    "products", expected_hits=0,
)

N1 | negative test - term on analyzed text
  Requirement:
    Try to match the exact string 'Rahul' with a term query on name.
  Query:
    {
      "term": {
        "name": "Rahul"
      }
    }
  Expected:
    0 hits
  Actual: 0 hit(s)
  Explanation:
    name is text: its terms are lowercase, so the capitalized literal does not exist in the index.
----------------------------------------------------------------------------------------------------


N2 | negative test - term case sensitivity on keyword
  Requirement:
    Try to match the category 'electronics' in lowercase.
  Query:
    {
      "term": {
        "category": "electronics"
      }
    }
  Expected:
    0 hits
  Actual: 0 hit(s)
  Explanation:
    keyword terms are case-sensitive; the indexed value is 'Electronics'.
----------------------------------------------------------------------------------------------------


ObjectApiResponse({'took': 14, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 0, 'relation': 'eq'}, 'max_score': None, 'hits': []}})

## Wrap-up

Key takeaways:

- Use `term`/`terms` on `keyword` fields and `match`/`match_phrase` on `text`.
- `range` queries are inclusive unless you use `gt`/`lt`; dates accept ISO
  strings and date math.
- `filter` context (inside `bool.filter` or at the top level with
  `constant_score`) skips scoring and is faster/cacheable - use it for exact
  business constraints.
- `multi_match`, `query_string` and `simple_query_string` are convenience
  layers over `bool` queries; `simple_query_string` is safest for raw user
  input.
- `fuzzy`, leading `wildcard` and `regexp` are powerful but expensive; prefer
  `prefix` or n-gram strategies at scale.
- Paginate with `from`/`size` for shallow pages; use `search_after` beyond the
  10 000-document `max_result_window`.
- Project with `source_includes`/`source_excludes` to keep responses small, and
  use `highlight` to explain matches to users.

The notebook is idempotent: re-running it recreates both indices and reloads
the datasets from `employees.json` and `products.json`.